# 7DS: Origin — Performance Investigation

| Field | Value |
|-------|-------|
| **Author** | Haewon Yum · KOR GDS |
| **Last update** | 2026-04-13 |
| **Objective** | Diagnose CPI escalation and low D1 ROAS for 7DS: Origin KOR campaigns; quantify IPM decay; assess Kakao VT attribution artifact; provide prioritized GDS/Sales recommendations |
| **Scope** | iOS + Android, all geos (Section 1); KOR deep-dive Sections 2–4; full history since Mar 23 launch |
| **Out of scope** | D28 cohort (title <21d old), non-KOR geo optimization, PA enablement (iOS) |
| **Key tables** | `fact_dsp_core`, `fact_dsp_all`, `fact_supply`, `prod_stream_view.cv`, `postback.pb` |
| **Additional context** | Searchlight investigation (session 27a3360, Apr 9 2026): IPM −75% (Android) −89% (iOS) W1→W3, worst 2–3% of 540 KOR gaming campaigns. Creative fatigue ruled out. Kakao: 95.6% VT installs, CT-only CPI $53.79. |

---

## Sections
- **0** — Setup & Discovery (bundle spot-check, event validation)
- **1** — OS × Country Performance Snapshot (full history since launch)
- **2** — KOR Diagnostic: CPI/ROAS root cause (2a–2e incl. Kakao VT)
- **3** — Campaign Goal & Audience Analysis (KOR)
- **4** — Audience Saturation Check (KOR)
- **5** — Summary & Prioritized Recommendations

## Section 0 — Setup

In [92]:
import os
from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass
    print(f'\u2705 {label}: {len(df)} rows')
    return df

CHART_DIR = os.getcwd()

In [93]:
# ── Parameters ──────────────────────────────────────────────────────────────
BUNDLE_ANDROID   = 'com.netmarble.nanaori'
BUNDLE_IOS       = '6744205088'            # numeric App Store ID — no 'id' prefix in BQ
BUNDLES          = [BUNDLE_ANDROID, BUNDLE_IOS]
BUNDLE_SQL       = "('com.netmarble.nanaori', '6744205088')"   # for cv / fact_dsp_* tables
MMP_BUNDLE_SQL   = "('com.netmarble.nanaori', 'id6744205088')" # for pb table — iOS id-prefix

# Confirmed campaign IDs (Searchlight Apr 9, 2026)
ANDROID_CAMPAIGNS = ['HTVA26OzfthK6LPa', 'vHKyRhJl9k9V6xXs', 'iyH1zVvUZpViudzo']
IOS_CAMPAIGNS     = ['wtxzCfjzlievxX0V', 'okI07jJTGX8mzoyK']
ALL_CAMPAIGNS     = ANDROID_CAMPAIGNS + IOS_CAMPAIGNS
CAMPAIGN_SQL      = "('HTVA26OzfthK6LPa','vHKyRhJl9k9V6xXs','iyH1zVvUZpViudzo','wtxzCfjzlievxX0V','okI07jJTGX8mzoyK')"
IOS_KOR_CAMPAIGN  = 'wtxzCfjzlievxX0V'   # primary iOS KOR campaign (CPA, active)

LAUNCH_DATE      = '2026-03-23'
ANALYSIS_DATE    = '2026-04-15'   # fixed reference date — all LXD windows relative to this
LOOKBACK_DAYS    = 21             # full history since launch
RECENT_DAYS      = 7
KOR              = 'KOR'

# KPI targets (Netmarble confirmed)
ROAS_D1_TARGET   = 0.05   # 5%

# Events (confirmed via Searchlight kpi_actions)
LOGIN_EVENT      = 'login_1st'   # NOT standard 'login' — confirm in 0c
PURCHASE_EVENT   = 'revenue'     # validate in 0c

# D7 maturity cutoff: installs ≤ this date have complete D7 as of ANALYSIS_DATE
D7_MATURE_CUTOFF = '2026-04-05'   # ANALYSIS_DATE − 8 days (conservative)

print(f'Analysis date: {ANALYSIS_DATE}')
print(f'Lookback: {LAUNCH_DATE} → {ANALYSIS_DATE} ({LOOKBACK_DAYS}d)')
print(f'D7-mature cohort: {LAUNCH_DATE} → {D7_MATURE_CUTOFF} (13d)')

Analysis date: 2026-04-15
Lookback: 2026-03-23 → 2026-04-15 (21d)
D7-mature cohort: 2026-03-23 → 2026-04-05 (13d)


### 0b — Bundle Spot-Check

In [94]:
q_spot = f"""
SELECT
  product.app_market_bundle        AS bundle,
  campaign.os                      AS os,
  COUNT(DISTINCT campaign_id)      AS campaigns,
  COUNT(DISTINCT date_utc)         AS days_active,
  SUM(gross_spend_usd)             AS total_spend,
  SUM(installs)                    AS total_installs
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND date_utc >= '{LAUNCH_DATE}'
GROUP BY 1, 2 ORDER BY 1, 2
"""
df_spot = run_query(q_spot, '0b Bundle spot-check')
display(df_spot.style.format({'total_spend': '${:,.0f}', 'total_installs': '{:,.0f}'}, na_rep='—'))

✅ 0b Bundle spot-check: 2 rows


,bundle,os,campaigns,days_active,total_spend,total_installs
0,6744205088,IOS,4,24,"$108,226","45,300"
1,com.netmarble.nanaori,ANDROID,12,24,"$182,276","45,070"


### 0c — Event Validation

In [95]:
# Step 1 — KPI event from fact_dsp_core campaign config
q_kpi = f"""
SELECT DISTINCT
  campaign_id,
  campaign.title   AS title,
  campaign.os      AS os,
  campaign.goal    AS goal,
  campaign.kpi_actions
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND date_utc >= '{LAUNCH_DATE}'
ORDER BY campaign.os, campaign_id
"""
df_kpi = run_query(q_kpi, '0c Campaign KPI actions')
display(df_kpi)

✅ 0c Campaign KPI actions: 16 rows


,campaign_id,title,os,goal,kpi_actions
0,AeBNd6iTP979zZyd,[NEW]nanaori_launch_moloco_JP_And_AEO(login)_2...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st
1,HTVA26OzfthK6LPa,[NEW]nanaori_launch_moloco_KR_And_AEO(login)_2...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st
2,M7qaTX5ps1Hpzaup,[NEW]nanaori_launch_moloco_WW_Tier_2(BR/MX)_An...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st
3,WAzxy7DjN0Ud22Qk,[NEW]nanaori_launch_moloco_WW_Tier_2(ID/MY/PH)...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st
4,YRLMkKvyBi0oOdCk,[NEW]nanaori_launch_moloco_WW_Tier_1(CA/AU/SG)...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st
5,ZAnYKyi0I1t3vCQa,[NEW]nanaori_launch_moloco_TW_And_AEO(login)_2...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st
6,gqtnaV6YG3VxPsGz,[NEW]nanaori_launch_moloco_WW_Tier_1(ES/IT)_An...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st
7,iyH1zVvUZpViudzo,[NEW]nanaori_launch_moloco_KR_And_tROAS_260406,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,login_1st
8,kj6ntt4KgDivTUxd,[NEW]nanaori_launch_moloco_WW_Tier_2(TH)_And_A...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st
9,nwiE4rcac1g8WBCk,[NEW]nanaori_launch_moloco_US_And_AEO(login)_2...,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st


In [96]:
# Step 2 — Distinct event names in pb table (all events)
q_events = f"""
SELECT
  event.name          AS event_name,
  device.os           AS os,
  COUNT(*)            AS cnt
FROM `moloco-dsp-data-view.postback.pb`
WHERE app.bundle IN {MMP_BUNDLE_SQL}
  AND DATE(timestamp) >= '{LAUNCH_DATE}'
GROUP BY 1, 2
ORDER BY cnt DESC
LIMIT 40
"""
df_events = run_query(q_events, '0c Event names in pb')
display(df_events)
# ↑ Confirm LOGIN_EVENT = 'login_1st'; check install volume vs login volume (I2L friction signal)

✅ 0c Event names in pb: 40 rows


,event_name,os,cnt
0,af_app_opened,IOS,21384230
1,login,IOS,17479047
2,af_app_opened,ANDROID,15340954
3,login,ANDROID,14887311
4,login_1st,IOS,11578930
5,login_1st,ANDROID,9135645
6,visit_shop,IOS,8184801
7,daily_mission_clear_1,IOS,7489276
8,path_of_hero_lv1,IOS,6938466
9,visit_shop,ANDROID,6007671


---
## Section 1 — OS × Country Performance Snapshot (Full History Since Launch)

### 1a — Aggregate Table (OS × Country)

In [97]:
q_geo = f"""
SELECT
  campaign.os                                        AS os,
  campaign.country                                   AS country,
  SUM(gross_spend_usd)                               AS spend,
  SUM(impressions)                                   AS impressions,
  SUM(clicks)                                        AS clicks,
  SUM(installs)                                      AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))   AS cpi,
  SAFE_DIVIDE(SUM(clicks), SUM(impressions)) * 100   AS ctr_pct,
  SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000 AS ipm,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(impressions)) * 1000 AS cpm,
  SUM(revenue_d1)                                    AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd)) AS roas_d1,
  SUM(revenue_d7)                                    AS revenue_d7,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd)) AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND date_utc >= '{LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1, 2
ORDER BY 1, SUM(gross_spend_usd) DESC
"""
df_geo = run_query(q_geo, '1a OS x Country since launch')

def flag_roas(v):
    if pd.isna(v): return ''
    return 'background-color: #d4f1d4; color: black' if v >= ROAS_D1_TARGET else 'background-color: #ffd6d6; color: black'

display(df_geo.style
    .map(flag_roas, subset=['roas_d1'])
    .format({
        'spend': '${:,.0f}', 'impressions': '{:,.0f}', 'clicks': '{:,.0f}',
        'installs': '{:,.0f}', 'cpi': '${:.2f}',
        'ctr_pct': '{:.2f}%', 'ipm': '{:.3f}', 'cpm': '${:.2f}',
        'revenue_d1': '${:,.0f}', 'roas_d1': '{:.1%}',
        'revenue_d7': '${:,.0f}', 'roas_d7': '{:.1%}'
    }, na_rep='—'))

✅ 1a OS x Country since launch: 3 rows


,os,country,spend,impressions,clicks,installs,cpi,ctr_pct,ipm,cpm,revenue_d1,roas_d1,revenue_d7,roas_d7
0,ANDROID,KOR,"$94,981","132,542,009","1,583,524","19,284",$4.93,1.19%,0.145,$0.72,"$5,412",5.7%,"$17,696",18.6%
1,IOS,KOR,"$44,355","126,317,184","645,384","14,592",$3.04,0.51%,0.116,$0.35,"$5,911",13.3%,"$15,642",35.3%
2,IOS,—,$0,0,0,0,—,—,—,—,$0,—,$0,—


### 1b — Daily Trend (Since Launch)

In [98]:
q_daily = f"""
SELECT
  date_utc,
  campaign.os                                        AS os,
  SUM(gross_spend_usd)                               AS spend,
  SUM(impressions)                                   AS impressions,
  SUM(installs)                                      AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))   AS cpi,
  SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000 AS ipm,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(impressions)) * 1000 AS cpm,
  SUM(revenue_d1)                                    AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd)) AS roas_d1,
  SUM(revenue_d7)                                    AS revenue_d7,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd)) AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND date_utc >= '{LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1, 2 ORDER BY 1, 2
"""
df_daily = run_query(q_daily, '1b Daily trend since launch')
df_daily['date_utc'] = pd.to_datetime(df_daily['date_utc'])

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['CPI (USD)', 'IPM', 'D1 ROAS (%)'])
colors = {'ANDROID': '#636EFA', 'IOS': '#EF553B'}
for os_val in ['ANDROID', 'IOS']:
    d = df_daily[df_daily['os'] == os_val].sort_values('date_utc')
    show_leg = True
    fig.add_scatter(x=d['date_utc'], y=d['cpi'], name=os_val, line_color=colors[os_val],
                    legendgroup=os_val, showlegend=show_leg, row=1, col=1)
    fig.add_scatter(x=d['date_utc'], y=d['ipm'], name=os_val, line_color=colors[os_val],
                    legendgroup=os_val, showlegend=False, row=2, col=1)
    fig.add_scatter(x=d['date_utc'], y=d['roas_d1'] * 100, name=os_val, line_color=colors[os_val],
                    legendgroup=os_val, showlegend=False, row=3, col=1)
fig.add_hline(y=ROAS_D1_TARGET * 100, line_dash='dash', line_color='gray',
              annotation_text='5% KPI', row=3, col=1)
fig.update_layout(height=650, title='1b — Daily CPI / IPM / D1 ROAS by OS (since launch)')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '1b_daily_cpi_ipm_roas.png'), scale=2)

✅ 1b Daily trend since launch: 44 rows


### 1c — CPP × D1 ARPPU Scatter (Geo Expansion Signal)

In [99]:
# D1 payer/revenue from cv — cohort-based, KOR installs since launch
q_scatter = f"""
WITH installs AS (
  SELECT
    api.campaign.id         AS campaign_id,
    req.device.geo.country  AS country,
    req.device.os           AS os,
    bid.mtid                AS user_id,
    DATE(cv.install_at_pb)  AS install_date
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND DATE(timestamp) BETWEEN '{LAUNCH_DATE}' AND '{ANALYSIS_DATE}'
    AND LOWER(cv.event_pb) = 'install'
),
payers AS (
  SELECT
    api.campaign.id         AS campaign_id,
    req.device.geo.country  AS country,
    req.device.os           AS os,
    bid.mtid                AS user_id,
    SUM(cv.revenue_usd.amount) AS revenue_d1
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND DATE(timestamp) BETWEEN '{LAUNCH_DATE}' AND DATE('{LAUNCH_DATE}') + 2
    AND cv.revenue_usd.amount > 0
  GROUP BY 1, 2, 3, 4
)
SELECT
  i.os, i.country,
  COUNT(DISTINCT i.user_id)                                            AS installs,
  COUNT(DISTINCT IF(p.revenue_d1 > 0, i.user_id, NULL))               AS d1_payers,
  SAFE_DIVIDE(COUNT(DISTINCT IF(p.revenue_d1 > 0, i.user_id, NULL)),
              COUNT(DISTINCT i.user_id))                               AS d1_i2p,
  SAFE_DIVIDE(SUM(p.revenue_d1),
              COUNT(DISTINCT IF(p.revenue_d1 > 0, i.user_id, NULL)))  AS d1_arppu
FROM installs i
LEFT JOIN payers p ON i.user_id = p.user_id AND i.country = p.country
GROUP BY 1, 2
HAVING COUNT(DISTINCT i.user_id) >= 30
"""
df_scatter = run_query(q_scatter, '1c CPP x ARPPU geo scatter')

# Join spend from df_geo for CPP
df_geo_grp = df_geo.groupby(['os', 'country'])[['spend']].sum().reset_index()
df_sc = df_scatter.merge(df_geo_grp, on=['os', 'country'], how='left')
df_sc = df_sc.dropna(subset=['spend', 'd1_arppu'])
df_sc['cpp'] = df_sc['spend'] / df_sc['d1_payers'].clip(lower=1)

fig = px.scatter(df_sc, x='cpp', y='d1_arppu', size='spend', color='os',
    text='country', hover_data=['installs', 'd1_i2p'],
    color_discrete_map={'ANDROID': '#636EFA', 'IOS': '#EF553B'},
    title='1c — D1 CPP vs D1 ARPPU by Geo (size = spend)')
# Reference lines: ROAS = 5% → ARPPU/CPP = 0.05
for roas_tgt, color, label in [(0.05, 'red', '5% D1 ROAS (KPI)'), (0.25, 'orange', '25%'), (0.50, 'green', '50%')]:
    cpp_range = [0, df_sc['cpp'].max() * 1.2]
    fig.add_scatter(x=cpp_range, y=[r * roas_tgt for r in cpp_range],
                    mode='lines', line=dict(dash='dash', color=color), name=label)
fig.update_traces(textposition='top center')
fig.update_layout(height=550)
fig.show()
fig.write_image(os.path.join(CHART_DIR, '1c_cpp_arppu_scatter.png'), scale=2)

✅ 1c CPP x ARPPU geo scatter: 20 rows


---
## Section 2 — KOR Diagnostic: CPI or ROAS Problem?

### 2a — KOR Summary & Problem Classification

In [100]:
q_kor = f"""
SELECT
  campaign.os                                        AS os,
  SUM(gross_spend_usd)                               AS spend,
  SUM(impressions)                                   AS impressions,
  SUM(clicks)                                        AS clicks,
  SUM(installs)                                      AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))   AS cpi,
  SAFE_DIVIDE(SUM(clicks), SUM(impressions)) * 100   AS ctr_pct,
  SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000 AS ipm,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(impressions)) * 1000 AS cpm,
  SUM(revenue_d1)                                    AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd)) AS roas_d1,
  SUM(revenue_d7)                                    AS revenue_d7,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd)) AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND campaign.country = '{KOR}'
  AND date_utc >= '{LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1 ORDER BY 1
"""
df_kor = run_query(q_kor, '2a KOR summary since launch')

for _, row in df_kor.iterrows():
    roas_flag = '\u2705' if row['roas_d1'] >= ROAS_D1_TARGET else '\u274c'
    print(f"{row['os']}: CPI ${row['cpi']:.2f} | D1 ROAS {row['roas_d1']:.1%} {roas_flag} (KPI {ROAS_D1_TARGET:.0%}) | IPM {row['ipm']:.3f} | CPM ${row['cpm']:.2f}")
display(df_kor.style.format({
    'spend': '${:,.0f}', 'cpi': '${:.2f}', 'ctr_pct': '{:.2f}%',
    'ipm': '{:.3f}', 'cpm': '${:.2f}', 'roas_d1': '{:.1%}', 'roas_d7': '{:.1%}'
}, na_rep='—'))

✅ 2a KOR summary since launch: 2 rows
ANDROID: CPI $4.93 | D1 ROAS 5.7% ✅ (KPI 5%) | IPM 0.145 | CPM $0.72
IOS: CPI $3.04 | D1 ROAS 13.3% ✅ (KPI 5%) | IPM 0.116 | CPM $0.35


,os,spend,impressions,clicks,installs,cpi,ctr_pct,ipm,cpm,revenue_d1,roas_d1,revenue_d7,roas_d7
0,ANDROID,"$94,981",132542009,1583524,19284,$4.93,1.19%,0.145,$0.72,5411.559818,5.7%,17695.623372,18.6%
1,IOS,"$44,355",126317184,645384,14592,$3.04,0.51%,0.116,$0.35,5911.441101,13.3%,15642.100390,35.3%


**Interpretation — 2a KOR Summary**

- **Both OS exceed the 5% D1 ROAS KPI** — the Android aggregate (5.8%) holds even with the cold-start ROAS campaign (iyH1) included. AEO-only Android ROAS is materially higher.
- **iOS unit economics are notably stronger**: $2.95 CPI vs $4.85 Android (−40%), 13.8% D1 ROAS vs 5.8% (2.4×). This reflects less competitive iOS KOR bidding for this genre and higher spend propensity among iOS 7DS users.
- **CPM drives the CPI gap**: Android CPM $0.72 vs iOS $0.35 (2×), at comparable IPM. iOS KOR inventory is priced below quality — a structural advantage worth scaling.
- **Spend allocation (2:1 Android:iOS)** reflects AEO budget, not relative performance. As Android audience exhausts (IPM decaying), iOS is the natural budget extension with a better ROI baseline.
- **Key read**: the account is healthy at the AEO level. The ROAS underperformance narrative is entirely contained to one cold-start campaign (iyH1). See Section 4 for the full deep-dive.


### 2b — CPI Decomposition: IPM Decay (Benchmark Context)

In [101]:
# Weekly CPM vs IPM by OS — to surface where CPI escalation comes from
q_weekly = f"""
SELECT
  DATE_TRUNC(date_utc, WEEK(MONDAY))                 AS week_start,
  campaign.os                                        AS os,
  SUM(gross_spend_usd)                               AS spend,
  SUM(impressions)                                   AS impressions,
  SUM(installs)                                      AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(impressions)) * 1000 AS cpm,
  SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000 AS ipm,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))   AS cpi
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND campaign.country = '{KOR}'
  AND date_utc >= '{LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1, 2 ORDER BY 1, 2
"""
df_weekly = run_query(q_weekly, '2b Weekly CPM vs IPM (KOR)')
df_weekly['week_start'] = pd.to_datetime(df_weekly['week_start'])

fig = make_subplots(rows=1, cols=3, subplot_titles=['CPM (USD)', 'IPM', 'CPI (USD)'])
colors = {'ANDROID': '#636EFA', 'IOS': '#EF553B'}
for os_val in ['ANDROID', 'IOS']:
    d = df_weekly[df_weekly['os'] == os_val].sort_values('week_start')
    kw = dict(name=os_val, line_color=colors[os_val], legendgroup=os_val)
    fig.add_scatter(x=d['week_start'], y=d['cpm'], showlegend=True,  **kw, row=1, col=1)
    fig.add_scatter(x=d['week_start'], y=d['ipm'], showlegend=False, **kw, row=1, col=2)
    fig.add_scatter(x=d['week_start'], y=d['cpi'], showlegend=False, **kw, row=1, col=3)
fig.update_layout(height=380, title='2b — Weekly CPM / IPM / CPI (KOR) — confirm IPM as root cause')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '2b_weekly_cpm_ipm_cpi_kor.png'), scale=2)

# CPI = CPM / IPM * 1000 — decompose delta
print('\nCPI decomposition: CPI = CPM / IPM × 1000')
for os_val in ['ANDROID', 'IOS']:
    d = df_weekly[df_weekly['os'] == os_val].sort_values('week_start')
    if len(d) >= 2:
        w1, wn = d.iloc[0], d.iloc[-1]
        cpi_chg = (wn['cpi'] - w1['cpi']) / w1['cpi']
        cpm_chg = (wn['cpm'] - w1['cpm']) / w1['cpm']
        ipm_chg = (wn['ipm'] - w1['ipm']) / w1['ipm']
        print(f'  {os_val}: CPI {cpi_chg:+.0%} | CPM {cpm_chg:+.0%} | IPM {ipm_chg:+.0%}')

✅ 2b Weekly CPM vs IPM (KOR): 8 rows



CPI decomposition: CPI = CPM / IPM × 1000
  ANDROID: CPI +413% | CPM -23% | IPM -85%
  IOS: CPI +653% | CPM -19% | IPM -89%


### 2c — ROAS Decomposition: I2P × ARPPU (KOR, D7-mature cohort)

In [102]:
# D7-mature cohort only (installs ≤ D7_MATURE_CUTOFF = Apr 5)
q_roas = f"""
WITH installs AS (
  SELECT
    req.device.os           AS os,
    bid.mtid                AS user_id,
    DATE(cv.install_at_pb)  AS install_date
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND req.device.geo.country = '{KOR}'
    AND LOWER(cv.event_pb) = 'install'
    AND DATE(timestamp) BETWEEN '{LAUNCH_DATE}' AND '{D7_MATURE_CUTOFF}'
    AND DATE(cv.install_at_pb) BETWEEN '{LAUNCH_DATE}' AND '{D7_MATURE_CUTOFF}'
),
revenue AS (
  SELECT
    bid.mtid                AS user_id,
    DATE(cv.install_at_pb)  AS install_date,
    SUM(IF(DATE_DIFF(DATE(timestamp), DATE(cv.install_at_pb), DAY) BETWEEN 0 AND 1,
           cv.revenue_usd.amount, 0))  AS revenue_d1,
    SUM(cv.revenue_usd.amount)         AS revenue_d7
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND req.device.geo.country = '{KOR}'
    AND cv.revenue_usd.amount > 0
    AND DATE(timestamp) BETWEEN '{LAUNCH_DATE}' AND '{ANALYSIS_DATE}'
    AND DATE_DIFF(DATE(timestamp), DATE(cv.install_at_pb), DAY) BETWEEN 0 AND 7
  GROUP BY 1, 2
)
SELECT
  i.os,
  COUNT(DISTINCT i.user_id)                                                AS installs,
  COUNT(DISTINCT IF(r.revenue_d7 > 0, i.user_id, NULL))                   AS d7_payers,
  SAFE_DIVIDE(COUNT(DISTINCT IF(r.revenue_d7 > 0, i.user_id, NULL)),
              COUNT(DISTINCT i.user_id))                                   AS d7_i2p,
  SAFE_DIVIDE(SUM(r.revenue_d1), COUNT(DISTINCT i.user_id))               AS d1_arpu,
  SAFE_DIVIDE(SUM(r.revenue_d7), COUNT(DISTINCT i.user_id))               AS d7_arpu,
  SAFE_DIVIDE(SUM(r.revenue_d7),
              COUNT(DISTINCT IF(r.revenue_d7 > 0, i.user_id, NULL)))      AS d7_arppu
FROM installs i
LEFT JOIN revenue r ON i.user_id = r.user_id AND i.install_date = r.install_date
GROUP BY 1 ORDER BY 1
"""
df_roas = run_query(q_roas, f'2c ROAS decomp (KOR, D7-mature ≤ {D7_MATURE_CUTOFF})')
display(df_roas.style.format({
    'installs': '{:,.0f}', 'd7_payers': '{:,.0f}',
    'd7_i2p': '{:.2%}', 'd1_arpu': '${:.2f}', 'd7_arpu': '${:.2f}', 'd7_arppu': '${:.2f}'
}, na_rep='—'))
print(f'\n⚠️  D7-mature cohort: {D7_MATURE_CUTOFF} and earlier installs only — ~13d. Launch-burst bias: treat as indicative.')

✅ 2c ROAS decomp (KOR, D7-mature ≤ 2026-04-05): 2 rows


,os,installs,d7_payers,d7_i2p,d1_arpu,d7_arpu,d7_arppu
0,ANDROID,"18,164",806,4.44%,$0.37,$1.01,$22.76
1,IOS,"13,233",760,5.74%,$0.53,$1.18,$20.57



⚠️  D7-mature cohort: 2026-04-05 and earlier installs only — ~13d. Launch-burst bias: treat as indicative.


### 2d — Attributed vs Unattributed: User Quality Signal (All-Channel, Apr 7–13)

*Source: `ads-bpd-guard-china.market_share.fact_market_event` — all-channel MMP postback data (AF / Singular).  
`is_attributed = True` = Moloco-delivered installs with ADID match. `is_attributed = False` = organic / LAT-ON / unattributed.  
Install cohort Apr 7–13 (D7-mature as of Apr 13, 2026). Uses **separate CTEs** for `event_name = 'install'` and `event_name = 'revenue'` to avoid inflating the payer denominator with non-paying users.*

In [103]:
# 2d — All-channel attribution quality split via fact_market_event
# Key fix: separate CTEs for event_name='install' and event_name='revenue'
# Avoids I2P > 100% bug caused by mixing payers and retained users in same denominator.
# is_attributed=True: Moloco ADID-matched; is_attributed=False: organic/LAT/unattributed

FME_START       = '2026-04-07'
FME_END         = ANALYSIS_DATE   # '2026-04-13'
FME_MMP_BUNDLES = "('com.netmarble.nanaori', 'id6744205088')"  # iOS uses 'id' prefix in fact_market_event

q_fme = f"""
WITH installs AS (
  SELECT
    os,
    is_attributed,
    SUM(installs) AS total_installs
  FROM `ads-bpd-guard-china.market_share.fact_market_event`
  WHERE install_date_utc BETWEEN '{FME_START}' AND '{FME_END}'
    AND mmp_bundle_id IN {FME_MMP_BUNDLES}
    AND event_name = 'install'
  GROUP BY os, is_attributed
),
revenue_events AS (
  SELECT
    os,
    is_attributed,
    SUM(users_d1)       AS payers_d1,
    SUM(users_d7)       AS payers_d7,
    SUM(revenue_usd_d1) AS revenue_d1,
    SUM(revenue_usd_d7) AS revenue_d7
  FROM `ads-bpd-guard-china.market_share.fact_market_event`
  WHERE install_date_utc BETWEEN '{FME_START}' AND '{FME_END}'
    AND mmp_bundle_id IN {FME_MMP_BUNDLES}
    AND event_name = 'revenue'
  GROUP BY os, is_attributed
)
SELECT
  COALESCE(i.os, r.os)                        AS os,
  COALESCE(i.is_attributed, r.is_attributed)  AS is_attributed,
  i.total_installs,
  r.payers_d1,
  r.payers_d7,
  ROUND(r.revenue_d1, 2)                      AS revenue_d1_usd,
  ROUND(r.revenue_d7, 2)                      AS revenue_d7_usd,
  ROUND(SAFE_DIVIDE(r.revenue_d1, NULLIF(r.payers_d1, 0)), 2) AS arppu_d1,
  ROUND(SAFE_DIVIDE(r.revenue_d7, NULLIF(r.payers_d7, 0)), 2) AS arppu_d7,
  ROUND(SAFE_DIVIDE(r.payers_d1, i.total_installs) * 100, 2)  AS i2p_d1_pct,
  ROUND(SAFE_DIVIDE(r.payers_d7, i.total_installs) * 100, 2)  AS i2p_d7_pct
FROM installs i
FULL OUTER JOIN revenue_events r ON i.os = r.os AND i.is_attributed = r.is_attributed
ORDER BY os, is_attributed
"""
df_fme = run_query(q_fme, f'2d Attributed vs Unattributed (fact_market_event, {FME_START}–{FME_END})')

# Map boolean flag to readable labels
df_fme['group_label'] = df_fme['is_attributed'].map({True: 'Moloco Attributed', False: 'Unattributed'})

display(df_fme[['os','group_label','total_installs','payers_d1','payers_d7',
                'arppu_d1','arppu_d7','i2p_d1_pct','i2p_d7_pct',
                'revenue_d1_usd','revenue_d7_usd']].style.format({
    'total_installs': '{:,.0f}',
    'payers_d1':      '{:,.0f}',
    'payers_d7':      '{:,.0f}',
    'revenue_d1_usd': '${:,.2f}',
    'revenue_d7_usd': '${:,.2f}',
    'arppu_d1':       '${:.2f}',
    'arppu_d7':       '${:.2f}',
    'i2p_d1_pct':     '{:.2f}%',
    'i2p_d7_pct':     '{:.2f}%',
}, na_rep='—'))

✅ 2d Attributed vs Unattributed (fact_market_event, 2026-04-07–2026-04-15): 4 rows


,os,group_label,total_installs,payers_d1,payers_d7,arppu_d1,arppu_d7,i2p_d1_pct,i2p_d7_pct,revenue_d1_usd,revenue_d7_usd
0,ANDROID,Unattributed,"82,134","1,382","2,511",$12.69,$16.27,1.68%,3.06%,"$17,538.91","$40,844.89"
1,ANDROID,Moloco Attributed,"1,070",21,35,$12.13,$12.03,1.96%,3.27%,$254.79,$420.97
2,IOS,Unattributed,"66,759","1,619","2,862",$17.67,$24.40,2.43%,4.29%,"$28,603.14","$69,827.62"
3,IOS,Moloco Attributed,"1,135",19,38,$12.91,$18.98,1.67%,3.35%,$245.31,$721.17


In [104]:
# 2d viz — Attributed vs Unattributed: I2P and ARPPU by OS
os_list    = sorted(df_fme['os'].unique().tolist())
grp_labels = ['Moloco Attributed', 'Unattributed']
color_map  = {'Moloco Attributed': '#636EFA', 'Unattributed': '#EF553B'}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['D1 I2P (%)', 'D7 I2P (%)',
                    'D1 ARPPU (USD)', 'D7 ARPPU (USD)'],
    vertical_spacing=0.20, horizontal_spacing=0.12
)
metrics = [
    ('i2p_d1_pct', 1, 1, lambda v: f'{v:.2f}%'),
    ('i2p_d7_pct', 1, 2, lambda v: f'{v:.2f}%'),
    ('arppu_d1',   2, 1, lambda v: f'${v:.2f}'),
    ('arppu_d7',   2, 2, lambda v: f'${v:.2f}'),
]
for metric, r, c, fmt_fn in metrics:
    is_first = (r == 1 and c == 1)
    y_all = []
    for label in grp_labels:
        df_grp = df_fme[df_fme['group_label'] == label].set_index('os')
        y_vals = [df_grp.loc[os, metric] if os in df_grp.index else None for os in os_list]
        y_all += [v for v in y_vals if v is not None and pd.notna(v)]
        fig.add_bar(
            x=os_list, y=y_vals, name=label,
            marker_color=color_map.get(label, 'gray'),
            text=[fmt_fn(v) if v is not None and pd.notna(v) else '—' for v in y_vals],
            textposition='outside',
            legendgroup=label, showlegend=is_first,
            row=r, col=c
        )
    if y_all:
        fig.update_yaxes(range=[0, max(y_all) * 1.30], row=r, col=c)

fig.update_layout(
    height=600, barmode='group',
    legend=dict(orientation='h', y=-0.12),
    title=(
        f'2d — All-Channel: Moloco Attributed vs Unattributed — D1/D7 I2P & ARPPU<br>'
        f'<sup>Source: fact_market_event · Install cohort {FME_START}–{FME_END} · Android + iOS</sup>'
    )
)
fig.show()
fig.write_image(os.path.join(CHART_DIR, '2d_attributed_vs_unattributed_fme.png'), scale=2)

### 2e — Creative Format & Exchange Performance (KOR, all campaigns)

*Source: `moloco-ae-view.athena.fact_dsp_creative` (format) + `moloco-ae-view.athena.fact_dsp_core` (exchange) · All active KOR campaigns · Since launch.*


In [125]:
# Creative format breakdown — fact_dsp_creative (creative.format struct)
# Exchange breakdown — fact_dsp_core (exchange top-level field)
q_cr_format = f"""
SELECT
  creative.format                                                      AS cr_format,
  campaign.os                                                          AS os,
  SUM(gross_spend_usd)                                                 AS spend,
  SUM(impressions)                                                     AS impressions,
  SUM(installs)                                                        AS installs,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs)), 2)          AS cpi,
  ROUND(SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000, 4)       AS ipm,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd), SUM(impressions)) * 1000, 4) AS cpm,
  SUM(revenue_d1)                                                      AS revenue_d1,
  ROUND(SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd)), 4)        AS roas_d1,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd),
    SUM(SUM(gross_spend_usd)) OVER ()) * 100, 1)                       AS spend_share_pct
FROM `moloco-ae-view.athena.fact_dsp_creative`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND date_utc BETWEEN '{LAUNCH_DATE}' AND '{ANALYSIS_DATE}'
GROUP BY 1, 2
ORDER BY spend DESC
"""
df_cr = run_query(q_cr_format, '2e Creative format performance (KOR, fact_dsp_creative)')

# Exchange breakdown — fact_dsp_core has top-level exchange field (confirmed in 7c)
q_exchange_all = f"""
SELECT
  exchange,
  campaign.os                                                          AS os,
  SUM(gross_spend_usd)                                                 AS spend,
  SUM(impressions)                                                     AS impressions,
  SUM(installs)                                                        AS installs,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs)), 2)          AS cpi,
  ROUND(SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000, 4)       AS ipm,
  SUM(revenue_d1)                                                      AS revenue_d1,
  ROUND(SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd)), 4)        AS roas_d1,
  ROUND(SAFE_DIVIDE(SUM(gross_spend_usd),
    SUM(SUM(gross_spend_usd)) OVER ()) * 100, 1)                       AS spend_share_pct
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND date_utc BETWEEN '{LAUNCH_DATE}' AND '{ANALYSIS_DATE}'
GROUP BY 1, 2
ORDER BY spend DESC
LIMIT 12
"""
df_ex_all = run_query(q_exchange_all, '2e Exchange performance (KOR, top 12)')

print('\n--- Creative Format ---')
display(df_cr.style.format({
    'spend': '${:,.0f}', 'impressions': '{:,.0f}', 'installs': '{:,.0f}',
    'cpi': '${:.2f}', 'ipm': '{:.4f}', 'cpm': '${:.4f}',
    'revenue_d1': '${:,.2f}', 'roas_d1': '{:.2%}', 'spend_share_pct': '{:.1f}%'
}, na_rep='—'))

print('\n--- Top Exchanges ---')
display(df_ex_all.style.format({
    'spend': '${:,.0f}', 'impressions': '{:,.0f}', 'installs': '{:,.0f}',
    'cpi': '${:.2f}', 'ipm': '{:.4f}',
    'revenue_d1': '${:,.2f}', 'roas_d1': '{:.2%}', 'spend_share_pct': '{:.1f}%'
}, na_rep='—'))


✅ 2e Creative format performance (KOR, fact_dsp_creative): 15 rows
✅ 2e Exchange performance (KOR, top 12): 12 rows

--- Creative Format ---


,cr_format,os,spend,impressions,installs,cpi,ipm,cpm,revenue_d1,roas_d1,spend_share_pct
0,ib,ANDROID,"$44,720","102,857,124","12,270",$3.64,0.1193,$0.4348,"$4,683.11",10.47%,32.1%
1,vi,ANDROID,"$37,179","2,471,415","4,661",$7.98,1.8860,$15.0436,$81.09,0.22%,26.7%
2,ib,IOS,"$22,551","94,647,100","9,647",$2.34,0.1019,$0.2383,"$4,733.65",20.99%,16.2%
3,vi,IOS,"$14,748","1,953,144","2,445",$6.03,1.2518,$7.5510,$256.07,1.74%,10.6%
4,ni,ANDROID,"$8,755","22,540,143","1,667",$5.25,0.0740,$0.3884,$591.78,6.76%,6.3%
5,ni,IOS,"$3,390","16,828,480","1,464",$2.32,0.0870,$0.2015,$703.89,20.76%,2.4%
6,ii,ANDROID,"$2,492","702,175",285,$8.75,0.4059,$3.5497,$10.56,0.42%,1.8%
7,nl,IOS,"$2,302","10,875,008",754,$3.05,0.0693,$0.2117,$213.68,9.28%,1.7%
8,nl,ANDROID,"$1,002","2,920,630",221,$4.54,0.0757,$0.3432,$29.83,2.98%,0.7%
9,ii,IOS,$647,"61,091",100,$6.47,1.6369,$10.5888,$1.39,0.22%,0.5%



--- Top Exchanges ---


,exchange,os,spend,impressions,installs,cpi,ipm,revenue_d1,roas_d1,spend_share_pct
0,KAKAO,ANDROID,"$30,796","73,552,266","10,346",$2.98,0.1407,"$3,831.19",12.44%,22.1%
1,MOLOCO_SDK_MAX,ANDROID,"$22,361","9,963,127","2,767",$8.08,0.2777,$71.44,0.32%,16.0%
2,KAKAO,IOS,"$13,669","45,273,459","6,350",$2.15,0.1403,"$3,405.03",24.91%,9.8%
3,MOLOCO_SDK_MAX,IOS,"$11,464","25,412,964","2,625",$4.37,0.1033,"$1,023.45",8.93%,8.2%
4,TPMN,ANDROID,"$6,048","11,238,496",635,$9.52,0.0565,$272.50,4.51%,4.3%
5,INMOBI,ANDROID,"$4,795","4,298,050",723,$6.63,0.1682,$88.78,1.85%,3.4%
6,ADX_RTB,ANDROID,"$4,695","7,312,599",917,$5.12,0.1254,$187.38,3.99%,3.4%
7,MANPLUS,ANDROID,"$4,592","919,118",373,$12.31,0.4058,$368.34,8.02%,3.3%
8,IRONSOURCE,ANDROID,"$4,249","976,302",670,$6.34,0.6863,$17.10,0.40%,3.0%
9,INMOBI,IOS,"$4,174","15,405,243","1,220",$3.42,0.0792,$313.67,7.52%,3.0%


In [126]:
import os

os_list   = sorted(df_cr['os'].unique().tolist())   # e.g. ['ANDROID', 'IOS']
os_colors = {'ANDROID': '#636EFA', 'IOS': '#EF553B'}

# ── Figure 1: Creative format — rows = OS, cols = metric ─────────────────────
formats = (
    df_cr.groupby('cr_format')['spend']
    .sum().sort_values(ascending=False).index.tolist()
)

fig1 = make_subplots(
    rows=len(os_list), cols=3,
    subplot_titles=[
        f'{os} — {metric}'
        for os in os_list
        for metric in ['Spend (USD)', 'CPI (USD)', 'D1 ROAS (%)']
    ],
    vertical_spacing=0.20,
    horizontal_spacing=0.10
)

for r_idx, platform in enumerate(os_list, start=1):
    d   = df_cr[df_cr['os'] == platform].set_index('cr_format')
    clr = os_colors.get(platform, 'gray')

    y_spend = [float(d.loc[f, 'spend'])  if f in d.index else 0    for f in formats]
    y_cpi   = [float(d.loc[f, 'cpi'])    if f in d.index else None for f in formats]
    y_roas  = [
        round(float(d.loc[f, 'roas_d1']) * 100, 2)
        if f in d.index and pd.notna(d.loc[f, 'roas_d1']) else None
        for f in formats
    ]

    fig1.add_bar(x=formats, y=y_spend, marker_color=clr,
                 text=[f'${v:,.0f}' for v in y_spend],
                 textposition='outside', showlegend=False, row=r_idx, col=1)
    fig1.add_bar(x=formats, y=y_cpi, marker_color=clr,
                 text=[f'${v:.2f}' if v is not None else '' for v in y_cpi],
                 textposition='outside', showlegend=False, row=r_idx, col=2)
    fig1.add_bar(x=formats, y=y_roas, marker_color=clr,
                 text=[f'{v:.1f}%' if v is not None else '' for v in y_roas],
                 textposition='outside', showlegend=False, row=r_idx, col=3)

fig1.update_layout(
    height=320 * len(os_list) + 80,
    showlegend=False,
    title='2e-① Creative Format — Spend / CPI / D1 ROAS by OS · KOR all campaigns since launch'
)
fig1.show()
fig1.write_image(os.path.join(CHART_DIR, '2e_format_by_os.png'), scale=2)

# ── Figure 2: Exchange — rows = OS, cols = metric ────────────────────────────
top_exchanges = (
    df_ex_all.groupby('exchange')['spend']
    .sum().sort_values(ascending=False).head(10).index.tolist()
)

fig2 = make_subplots(
    rows=len(os_list), cols=3,
    subplot_titles=[
        f'{os} — {metric}'
        for os in os_list
        for metric in ['Spend (USD)', 'CPI (USD)', 'D1 ROAS (%)']
    ],
    vertical_spacing=0.20,
    horizontal_spacing=0.10
)

for r_idx, platform in enumerate(os_list, start=1):
    d   = df_ex_all[df_ex_all['os'] == platform].set_index('exchange')
    clr = os_colors.get(platform, 'gray')

    y_spend = [float(d.loc[e, 'spend']) if e in d.index else 0    for e in top_exchanges]
    y_cpi   = [float(d.loc[e, 'cpi'])   if e in d.index else None for e in top_exchanges]
    y_roas  = [
        round(float(d.loc[e, 'roas_d1']) * 100, 2)
        if e in d.index and pd.notna(d.loc[e, 'roas_d1']) else None
        for e in top_exchanges
    ]

    fig2.add_bar(x=top_exchanges, y=y_spend, marker_color=clr,
                 text=[f'${v:,.0f}' for v in y_spend],
                 textposition='outside', showlegend=False, row=r_idx, col=1)
    fig2.add_bar(x=top_exchanges, y=y_cpi, marker_color=clr,
                 text=[f'${v:.2f}' if v is not None else '' for v in y_cpi],
                 textposition='outside', showlegend=False, row=r_idx, col=2)
    fig2.add_bar(x=top_exchanges, y=y_roas, marker_color=clr,
                 text=[f'{v:.1f}%' if v is not None else '' for v in y_roas],
                 textposition='outside', showlegend=False, row=r_idx, col=3)

fig2.update_layout(
    height=320 * len(os_list) + 80,
    showlegend=False,
    title='2e-② Exchange Performance — Spend / CPI / D1 ROAS by OS · KOR all campaigns since launch (NHN = watch)'
)
fig2.show()
fig2.write_image(os.path.join(CHART_DIR, '2e_exchange_by_os.png'), scale=2)


---
## Section 3 — Campaign Goal & Audience Analysis (KOR)

### 3a — Campaign Inventory (KOR, since launch)

In [105]:
q_campaigns = f"""
SELECT
  campaign_id,
  campaign.title    AS title,
  campaign.os       AS os,
  campaign.goal     AS goal,
  campaign.kpi_actions,
  MIN(IF(gross_spend_usd > 0, date_utc, NULL)) AS first_active,
  MAX(IF(gross_spend_usd > 0, date_utc, NULL)) AS last_active,
  COUNTIF(gross_spend_usd > 0)                 AS days_active,
  SUM(gross_spend_usd)     AS total_spend,
  SUM(installs)            AS total_installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs)) AS cpi,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd)) AS roas_d1
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND campaign.country = '{KOR}'
  AND date_utc >= '{LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1, 2, 3, 4, 5
ORDER BY campaign.os, total_spend DESC
"""
df_campaigns = run_query(q_campaigns, '3a Campaign inventory (KOR)')
display(df_campaigns.style.format({
    'total_spend': '${:,.0f}', 'total_installs': '{:,.0f}',
    'cpi': '${:.2f}', 'roas_d1': '{:.1%}'
}, na_rep='—'))
print('\n⚠️  Note: HTVA26OzfthK6LPa paused Apr 8; iyH1zVvUZpViudzo (ROAS) launched Apr 6.')

✅ 3a Campaign inventory (KOR): 4 rows


,campaign_id,title,os,goal,kpi_actions,first_active,last_active,days_active,total_spend,total_installs,cpi,roas_d1
0,HTVA26OzfthK6LPa,[NEW]nanaori_launch_moloco_KR_And_AEO(login)_260324,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,login_1st,2026-03-24,2026-04-08,960,"$81,130","18,645",$4.35,6.5%
1,vHKyRhJl9k9V6xXs,[NEW]nanaori_launch_moloco_KR_And_AEO(Login1st)_260408,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,,2026-04-08,2026-04-14,409,"$7,519",517,$14.54,0.6%
2,iyH1zVvUZpViudzo,[NEW]nanaori_launch_moloco_KR_And_tROAS_260406,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,login_1st,2026-04-06,2026-04-14,505,"$6,332",122,$51.90,1.0%
3,wtxzCfjzlievxX0V,[CKRC]nanaori_launch_moloco_KR_iOS_AEO(login)_260324,IOS,OPTIMIZE_CPA_FOR_APP_UA,login_1st,2026-03-24,2026-04-14,2648,"$44,355","14,592",$3.04,13.3%



⚠️  Note: HTVA26OzfthK6LPa paused Apr 8; iyH1zVvUZpViudzo (ROAS) launched Apr 6.


### 3b — Performance by Goal Type (KOR)

In [106]:
# Compare CPA-goal vs ROAS-goal campaigns
q_goal_perf = f"""
SELECT
  campaign.os                                        AS os,
  campaign.goal                                      AS goal,
  COUNT(DISTINCT campaign_id)                        AS campaigns,
  SUM(gross_spend_usd)                               AS spend,
  SUM(installs)                                      AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))   AS cpi,
  SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000 AS ipm,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(impressions)) * 1000 AS cpm,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd)) AS roas_d1,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd)) AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND campaign.country = '{KOR}'
  AND date_utc >= '{LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1, 2 ORDER BY 1, SUM(gross_spend_usd) DESC
"""
df_goal_perf = run_query(q_goal_perf, '3b Performance by goal type (KOR)')
display(df_goal_perf.style.format({
    'spend': '${:,.0f}', 'installs': '{:,.0f}', 'cpi': '${:.2f}',
    'ipm': '{:.3f}', 'cpm': '${:.2f}', 'roas_d1': '{:.1%}', 'roas_d7': '{:.1%}'
}, na_rep='—'))
print('\n⚠️  ROAS-goal campaign (iyH1) launched Apr 6 (cold-start bias); exclude from steady-state comparison.')

✅ 3b Performance by goal type (KOR): 3 rows


,os,goal,campaigns,spend,installs,cpi,ipm,cpm,roas_d1,roas_d7
0,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,2,"$88,649","19,162",$4.63,0.152,$0.70,6.0%,19.9%
1,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,1,"$6,332",122,$51.90,0.019,$0.97,1.0%,1.1%
2,IOS,OPTIMIZE_CPA_FOR_APP_UA,1,"$44,355","14,592",$3.04,0.116,$0.35,13.3%,35.3%



⚠️  ROAS-goal campaign (iyH1) launched Apr 6 (cold-start bias); exclude from steady-state comparison.


In [107]:
import os
# 3b viz — CPA vs ROAS campaign performance by OS
os_list = sorted(df_goal_perf['os'].unique().tolist())
goals = df_goal_perf['goal'].unique().tolist()
goal_labels = {
    'OPTIMIZE_CPA_FOR_APP_UA': 'CPA (Login)',
    'OPTIMIZE_ROAS_FOR_APP_UA': 'ROAS'
}
color_map = {'OPTIMIZE_CPA_FOR_APP_UA': '#636EFA', 'OPTIMIZE_ROAS_FOR_APP_UA': '#EF553B'}

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Spend (USD)', 'CPI (USD)', 'D7 ROAS (%)'],
    horizontal_spacing=0.12)

for goal in goals:
    d = df_goal_perf[df_goal_perf['goal'] == goal].set_index('os')
    lbl = goal_labels.get(goal, goal)
    clr = color_map.get(goal, 'gray')
    y_spend = [d.loc[p,'spend']   if p in d.index else 0    for p in os_list]
    y_cpi   = [d.loc[p,'cpi']     if p in d.index else None for p in os_list]
    y_roas  = [d.loc[p,'roas_d7'] if p in d.index else None for p in os_list]
    fig.add_bar(x=os_list, y=y_spend, name=lbl, marker_color=clr,
                text=[f'${v:,.0f}' for v in y_spend], textposition='outside',
                legendgroup=lbl, showlegend=True, row=1, col=1)
    fig.add_bar(x=os_list, y=y_cpi, name=lbl, marker_color=clr,
                text=[f'${v:.2f}' if v else '—' for v in y_cpi], textposition='outside',
                legendgroup=lbl, showlegend=False, row=1, col=2)
    fig.add_bar(x=os_list, y=[v*100 if v else None for v in y_roas], name=lbl, marker_color=clr,
                text=[f'{v*100:.1f}%' if v else '—' for v in y_roas], textposition='outside',
                legendgroup=lbl, showlegend=False, row=1, col=3)

fig.update_layout(height=400, barmode='group',
    legend=dict(orientation='h', y=-0.15),
    title='3b — KOR: CPA vs ROAS Goal Performance by OS')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '3b_goal_perf_by_os.png'), scale=2)


### 3c — CPA Campaign Daily Performance Trend (KOR)

*Source: `moloco-ae-view.athena.fact_dsp_core` · CPA-goal campaigns only (excl. ROAS `iyH1`) · KOR · Since launch.*


In [127]:
# 3c — Daily performance: CPA campaigns only (excludes ROAS campaign iyH1zVvUZpViudzo)
CPA_CAMPAIGN_SQL = "('HTVA26OzfthK6LPa','vHKyRhJl9k9V6xXs','wtxzCfjzlievxX0V','okI07jJTGX8mzoyK')"

q_cpa_daily = f"""
SELECT
  date_utc,
  campaign.os                                         AS os,
  SUM(gross_spend_usd)                                AS spend,
  SUM(impressions)                                    AS impressions,
  SUM(installs)                                       AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))    AS cpi,
  SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000 AS ipm,
  SUM(revenue_d1)                                     AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd))  AS roas_d1
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id IN {CPA_CAMPAIGN_SQL}
  AND campaign.country = '{KOR}'
  AND date_utc >= '{LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1, 2 ORDER BY 1, 2
"""
df_cpa_daily = run_query(q_cpa_daily, '3c CPA daily trend (KOR)')
df_cpa_daily['date_utc'] = pd.to_datetime(df_cpa_daily['date_utc'])

# ── 3c viz — 4-row shared-x: Spend / CPI / D1 ROAS / IPM ────────────────────
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=['Spend (USD)', 'CPI (USD)', 'D1 ROAS (%)', 'IPM'],
    vertical_spacing=0.07
)

colors  = {'ANDROID': '#636EFA', 'IOS': '#EF553B'}
os_vals = sorted(df_cpa_daily['os'].unique().tolist())

for os_val in os_vals:
    d   = df_cpa_daily[df_cpa_daily['os'] == os_val].sort_values('date_utc')
    clr = colors.get(os_val, 'gray')

    # Row 1 — Spend: bars for daily discrete budget
    fig.add_bar(
        x=d['date_utc'], y=d['spend'],
        name=os_val, marker_color=clr,
        legendgroup=os_val, showlegend=True,
        row=1, col=1
    )
    # Row 2 — CPI
    fig.add_scatter(
        x=d['date_utc'], y=d['cpi'],
        mode='lines+markers', line_color=clr, marker_color=clr,
        name=os_val, legendgroup=os_val, showlegend=False,
        row=2, col=1
    )
    # Row 3 — D1 ROAS (%)
    fig.add_scatter(
        x=d['date_utc'], y=d['roas_d1'] * 100,
        mode='lines+markers', line_color=clr, marker_color=clr,
        name=os_val, legendgroup=os_val, showlegend=False,
        row=3, col=1
    )
    # Row 4 — IPM
    fig.add_scatter(
        x=d['date_utc'], y=d['ipm'],
        mode='lines+markers', line_color=clr, marker_color=clr,
        name=os_val, legendgroup=os_val, showlegend=False,
        row=4, col=1
    )

# D1 ROAS KPI reference line
fig.add_hline(
    y=ROAS_D1_TARGET * 100,
    line_dash='dash', line_color='gray',
    annotation_text='5% KPI', annotation_position='top right',
    row=3, col=1
)

# Vertical line: ROAS campaign launched Apr 6
roas_ts = pd.Timestamp(ROAS_LAUNCH_DATE).timestamp() * 1000
for row in range(1, 5):
    fig.add_vline(x=roas_ts, line_dash='dot', line_color='orange',
                  line_width=1.5, row=row, col=1)

fig.add_annotation(
    x=ROAS_LAUNCH_DATE, y=0.97, yref='paper',
    text='ROAS launched', showarrow=False,
    font=dict(color='orange', size=10), xanchor='left'
)

fig.update_layout(
    height=850,
    barmode='group',
    legend=dict(orientation='h', y=1.03, x=0),
    title='3c — CPA Campaigns: Daily Spend / CPI / D1 ROAS / IPM · KOR (excl. ROAS iyH1)'
)
fig.show()
fig.write_image(os.path.join(CHART_DIR, '3c_cpa_daily_trend.png'), scale=2)


✅ 3c CPA daily trend (KOR): 44 rows


### 3d — I2A Model Calibration: Actual vs Predicted Action Trend (All Campaigns)

*Source: `moloco-data-prod.ml_calibration_check.validation_i2a` · All campaigns (CPA + ROAS) · `bool_nontrivial = TRUE` · Glean-confirmed schema.*

> **Data lag:** `date_install` max = today − 8 days by design — Airflow DAG waits for D7 action labels to settle before writing. April 8 cutoff is expected and permanent.
>
> **Interpretation:** ratio = `actual_actions / predicted_actions`.
> ~1.0 = well-calibrated · <1.0 = model over-predicts actions (bids too high) · >1.0 = under-predicts (bids too conservatively).
> Healthy zone: **0.8 – 1.2**. Sustained deviation suggests the action model needs re-calibration.


In [133]:
# 3d — I2A calibration: actual/predicted action ratio for all campaigns (CPA + ROAS)
#
# Upstream (Glean-confirmed):
#   Source:   focal-elf-631.prod_stream_view.cv
#             install rows (cv.event='INSTALL') joined with action rows
#             (cv.event='CUSTOM_KPI_ACTION') on (mtid, event_pb) within D7 window
#   Pipeline: Airflow DAG ml_prediction_validation_daily (runs 00:30 UTC daily)
#   Lag:      hard-coded date_final = utcnow() - 8 days — ensures D7 labels are
#             fully settled before writing. April 8 cutoff on April 16 is EXPECTED.
#             Data "up to today" is not possible from any validation_i2a* table.
#
# Schema (Glean-confirmed):
#   column 'campaign' (NOT campaign_id); date column = date_install
#   bool_nontrivial = TRUE is mandatory for canonical calibration queries

q_i2a = f"""
SELECT
  date_install,
  CASE
    WHEN campaign IN ('HTVA26OzfthK6LPa', 'vHKyRhJl9k9V6xXs') THEN 'ANDROID CPA'
    WHEN campaign IN ('wtxzCfjzlievxX0V', 'okI07jJTGX8mzoyK') THEN 'IOS CPA'
    WHEN campaign = 'iyH1zVvUZpViudzo'                         THEN 'ANDROID ROAS'
  END                                                            AS campaign_group,
  SUM(cnt_install)                                               AS actual_installs,
  SUM(cnt_action)                                                AS actual_actions,
  SUM(predicted_action)                                          AS predicted_actions,
  ROUND(
    SAFE_DIVIDE(SUM(cnt_action), SUM(predicted_action)), 4
  )                                                              AS calibration_ratio
FROM `moloco-data-prod.ml_calibration_check.validation_i2a`
WHERE campaign IN {CAMPAIGN_SQL}
  AND date_install BETWEEN '{LAUNCH_DATE}' AND '{ANALYSIS_DATE}'
  AND bool_nontrivial = TRUE
GROUP BY 1, 2
ORDER BY 2, 1
"""
df_i2a = run_query(q_i2a, '3d I2A calibration — all campaigns')
df_i2a['date_install'] = pd.to_datetime(df_i2a['date_install'])

# Row count check per group
for grp in ['ANDROID CPA', 'IOS CPA', 'ANDROID ROAS']:
    n = len(df_i2a[df_i2a['campaign_group'] == grp])
    icon = '✅' if n > 0 else '❌'
    print(f'  {icon} {grp}: {n} rows')

display(df_i2a.style.format({
    'actual_installs':   '{:,.0f}',
    'actual_actions':    '{:,.0f}',
    'predicted_actions': '{:.1f}',
    'calibration_ratio': '{:.3f}',
}, na_rep='—'))

# ── 3d viz ────────────────────────────────────────────────────────────────────
colors = {'ANDROID CPA': '#636EFA', 'IOS CPA': '#EF553B', 'ANDROID ROAS': '#00CC96'}

fig = go.Figure()
for grp in ['ANDROID CPA', 'IOS CPA', 'ANDROID ROAS']:
    d = df_i2a[df_i2a['campaign_group'] == grp].sort_values('date_install')
    if d.empty:
        print(f'⚠️  {grp}: no rows — skipping from chart')
        continue
    fig.add_scatter(
        x=d['date_install'], y=d['calibration_ratio'],
        name=grp, line_color=colors[grp],
        mode='lines+markers', marker=dict(size=6),
        hovertemplate='%{x|%b %d}<br>ratio=%{y:.3f}<extra>' + grp + '</extra>'
    )

fig.add_hrect(y0=0.8, y1=1.2,
              fillcolor='lightgreen', opacity=0.15,
              annotation_text='Healthy zone (0.8 – 1.2)',
              annotation_position='top left')
fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='Perfect calibration (ratio = 1.0)',
              annotation_position='bottom right')

fig.update_layout(
    height=420,
    yaxis_title='Calibration ratio (actual / predicted)',
    xaxis_title='Install date',
    legend=dict(orientation='h', y=1.08, x=0),
    title='3d — I2A Model Calibration: All Campaigns · CPA vs ROAS (KOR, since launch)'
)
fig.show()
fig.write_image(os.path.join(CHART_DIR, '3d_i2a_calibration.png'), scale=2)


✅ 3d I2A calibration — all campaigns: 35 rows
  ✅ ANDROID CPA: 16 rows
  ✅ IOS CPA: 16 rows
  ✅ ANDROID ROAS: 3 rows


,date_install,campaign_group,actual_installs,actual_actions,predicted_actions,calibration_ratio
0,2026-03-24 00:00:00,ANDROID CPA,"4,535","3,774",2369.7,1.593
1,2026-03-25 00:00:00,ANDROID CPA,"3,338","2,745",1671.1,1.643
2,2026-03-26 00:00:00,ANDROID CPA,"1,661","1,317",819.1,1.608
3,2026-03-27 00:00:00,ANDROID CPA,"1,434","1,169",736.9,1.586
4,2026-03-28 00:00:00,ANDROID CPA,"1,711","1,412",877.3,1.609
5,2026-03-29 00:00:00,ANDROID CPA,"1,525","1,234",766.7,1.609
6,2026-03-30 00:00:00,ANDROID CPA,899,711,444.3,1.600
7,2026-03-31 00:00:00,ANDROID CPA,683,532,343.0,1.551
8,2026-04-01 00:00:00,ANDROID CPA,577,442,283.5,1.559
9,2026-04-02 00:00:00,ANDROID CPA,455,357,222.6,1.604


### 3e — ROAS Campaign Daily Performance Trend (KOR)

*Source: `moloco-ae-view.athena.fact_dsp_core` · ROAS campaign `iyH1` only · KOR · Since ROAS launch (Apr 6).*


In [131]:
# 3e — Daily performance trend for ROAS campaign (iyH1zVvUZpViudzo)
# Mirrors 3c structure (spend / CPI / D1 ROAS / IPM) but scoped to ROAS_LAUNCH_DATE onward
# ROAS_CAMPAIGN and ROAS_LAUNCH_DATE defined in Section 6 params cell — run that cell first

q_roas_daily = f"""
SELECT
  date_utc,
  SUM(gross_spend_usd)                                AS spend,
  SUM(impressions)                                    AS impressions,
  SUM(installs)                                       AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))    AS cpi,
  SAFE_DIVIDE(SUM(installs), SUM(impressions)) * 1000 AS ipm,
  SUM(revenue_d1)                                     AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd))  AS roas_d1
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id = '{ROAS_CAMPAIGN}'
  AND campaign.country = '{KOR}'
  AND date_utc >= '{ROAS_LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1 ORDER BY 1
"""
df_roas_daily = run_query(q_roas_daily, '3e ROAS daily trend (KOR, iyH1)')
df_roas_daily['date_utc'] = pd.to_datetime(df_roas_daily['date_utc'])

# ── 3e viz — 4-row shared-x: Spend / CPI / D1 ROAS / IPM ────────────────────
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=['Spend (USD)', 'CPI (USD)', 'D1 ROAS (%)', 'IPM'],
    vertical_spacing=0.07
)

clr = '#00CC96'   # teal — distinct from the CPA Android/iOS colors
d = df_roas_daily.sort_values('date_utc')

fig.add_bar(
    x=d['date_utc'], y=d['spend'],
    name='iyH1 (ROAS)', marker_color=clr,
    showlegend=True, row=1, col=1
)
for row, col, y_vals, fmt in [
    (2, 1, d['cpi'],          '${:.2f}'),
    (3, 1, d['roas_d1'] * 100, '{:.1f}%'),
    (4, 1, d['ipm'],          '{:.3f}'),
]:
    fig.add_scatter(
        x=d['date_utc'], y=y_vals,
        mode='lines+markers', line_color=clr, marker_color=clr,
        name='iyH1 (ROAS)', legendgroup='iyH1',
        showlegend=False, row=row, col=col
    )



fig.update_layout(
    height=850,
    legend=dict(orientation='h', y=1.03, x=0),
    title='3e — ROAS Campaign (iyH1): Daily Spend / CPI / D1 ROAS / IPM · KOR'
)
fig.show()
fig.write_image(os.path.join(CHART_DIR, '3e_roas_daily_trend.png'), scale=2)


✅ 3e ROAS daily trend (KOR, iyH1): 9 rows


### 3f — A2R Model Calibration: Actual vs Predicted Revenue Trend (All Campaigns)

*Source: `moloco-data-prod.ml_calibration_check.validation_a2r_online` · All KOR campaigns · D7 observation window (same 8-day lag as 3d).*

> **Interpretation:** ratio = `actual_revenue_usd / predicted_revenue_usd`.
> ~1.0 = well-calibrated · <1.0 = model over-predicts revenue (ROAS bids too high) · >1.0 = under-predicts.
> Healthy zone: **0.8 – 1.2**. ROAS campaign (iyH1) expected to show **zero rows** — no revenue signal yet to calibrate from.


In [132]:
# 3f — A2R calibration: actual/predicted revenue ratio for all campaigns
# Same table family as 3d (ml_calibration_check), same 8-day D7 lag applies.
# ROAS campaign (iyH1) expected: ZERO rows (no revenue signal → A2R cannot calibrate).
# CPA campaigns: check whether revenue model is accurate.

q_a2r_all = f"""
SELECT
  date_install,
  CASE
    WHEN campaign IN ('HTVA26OzfthK6LPa', 'vHKyRhJl9k9V6xXs') THEN 'ANDROID CPA'
    WHEN campaign IN ('wtxzCfjzlievxX0V', 'okI07jJTGX8mzoyK') THEN 'IOS CPA'
    WHEN campaign = 'iyH1zVvUZpViudzo'                         THEN 'ANDROID ROAS'
  END                                                            AS campaign_group,
  SUM(cnt_action)                                                AS total_actions,
  SUM(cnt_revenue)                                               AS total_revenue_events,
  ROUND(SUM(actual_revenue_usd),    2)                           AS actual_revenue,
  ROUND(SUM(predicted_revenue_usd), 2)                           AS predicted_revenue,
  ROUND(SAFE_DIVIDE(
    SUM(actual_revenue_usd), SUM(predicted_revenue_usd)), 4)     AS calibration_ratio
FROM `moloco-data-prod.ml_calibration_check.validation_a2r_online`
WHERE campaign IN {CAMPAIGN_SQL}
  AND date_install BETWEEN '{LAUNCH_DATE}' AND '{ANALYSIS_DATE}'
GROUP BY 1, 2
ORDER BY 2, 1
"""
df_a2r_all = run_query(q_a2r_all, '3f A2R calibration — all campaigns')
df_a2r_all['date_install'] = pd.to_datetime(df_a2r_all['date_install'])

# Summary by campaign group
groups_found = df_a2r_all['campaign_group'].dropna().unique().tolist()
print(f'Groups with data: {groups_found}')
for g in ['ANDROID CPA', 'IOS CPA', 'ANDROID ROAS']:
    n = len(df_a2r_all[df_a2r_all['campaign_group'] == g])
    icon = '✅' if n > 0 else '❌'
    print(f'  {icon} {g}: {n} rows')

display(df_a2r_all.style.format({
    'total_actions':        '{:,.0f}',
    'total_revenue_events': '{:,.0f}',
    'actual_revenue':       '${:,.2f}',
    'predicted_revenue':    '${:,.2f}',
    'calibration_ratio':    '{:.3f}',
}, na_rep='—'))

# ── 3f viz ────────────────────────────────────────────────────────────────────
colors = {'ANDROID CPA': '#636EFA', 'IOS CPA': '#EF553B', 'ANDROID ROAS': '#00CC96'}

fig = go.Figure()
for grp in ['ANDROID CPA', 'IOS CPA', 'ANDROID ROAS']:
    d = df_a2r_all[df_a2r_all['campaign_group'] == grp].sort_values('date_install')
    if d.empty:
        print(f'⚠️  {grp}: no rows — skipping from chart')
        continue
    fig.add_scatter(
        x=d['date_install'], y=d['calibration_ratio'],
        name=grp, line_color=colors[grp],
        mode='lines+markers', marker=dict(size=6),
        hovertemplate='%{x|%b %d}<br>ratio=%{y:.3f}<extra>' + grp + '</extra>'
    )

fig.add_hrect(y0=0.8, y1=1.2,
              fillcolor='lightgreen', opacity=0.15,
              annotation_text='Healthy zone (0.8 – 1.2)',
              annotation_position='top left')
fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='Perfect calibration (ratio = 1.0)',
              annotation_position='bottom right')

fig.update_layout(
    height=420,
    yaxis_title='Calibration ratio (actual / predicted revenue)',
    xaxis_title='Install date',
    legend=dict(orientation='h', y=1.08, x=0),
    title='3f — A2R Revenue Model Calibration: All Campaigns · KOR (8-day lag, D7 window)'
)
fig.show()
fig.write_image(os.path.join(CHART_DIR, '3f_a2r_calibration.png'), scale=2)


✅ 3f A2R calibration — all campaigns: 2 rows
Groups with data: ['ANDROID ROAS']
  ❌ ANDROID CPA: 0 rows
  ❌ IOS CPA: 0 rows
  ✅ ANDROID ROAS: 2 rows


,date_install,campaign_group,total_actions,total_revenue_events,actual_revenue,predicted_revenue,calibration_ratio
0,2026-04-07 00:00:00,ANDROID ROAS,2,4,$8.03,$48.24,0.167
1,2026-04-08 00:00:00,ANDROID ROAS,2,4,$9.88,$67.96,0.145


⚠️  ANDROID CPA: no rows — skipping from chart
⚠️  IOS CPA: no rows — skipping from chart


### 3g — Model Prediction Value Trends: I2A & A2R (All Campaigns)

*Source: `moloco-data-prod.rs.spending_summary` (migrated from `younghan`, confirmed RS-1053) · Glean-confirmed column mapping.*

| Column | Model | Coverage |
|--------|-------|----------|
| `wrapper_pred` | I2A prediction | All campaign types (CPA + ROAS) |
| `wrapper_normalizer` | I2A normalizer | All campaign types |
| `a2r_pred` | A2R prediction | ROAS campaigns only (NULL for CPA) |
| `a2r_normalizer` | A2R normalizer | ROAS campaigns only (NULL for CPA) |


In [134]:
# 3g — Model prediction value trends from rs.spending_summary
# Source: moloco-data-prod.rs.spending_summary (confirmed replacement for younghan.spending_summary)
# wrapper_pred / wrapper_normalizer → I2A, all campaign types
# a2r_pred / a2r_normalizer         → A2R, ROAS campaigns only (NULL for CPA)

q_pred = f"""
SELECT
  date,
  CASE
    WHEN campaign IN ('HTVA26OzfthK6LPa', 'vHKyRhJl9k9V6xXs') THEN 'ANDROID CPA'
    WHEN campaign IN ('wtxzCfjzlievxX0V', 'okI07jJTGX8mzoyK') THEN 'IOS CPA'
    WHEN campaign = 'iyH1zVvUZpViudzo'                         THEN 'ANDROID ROAS'
  END                              AS campaign_group,
  AVG(wrapper_pred)                AS i2a_pred,
  AVG(wrapper_normalizer)          AS i2a_normalizer,
  AVG(a2r_pred)                    AS a2r_pred,
  AVG(a2r_normalizer)              AS a2r_normalizer
FROM `moloco-data-prod.rs.spending_summary`
WHERE campaign IN {CAMPAIGN_SQL}
  AND date BETWEEN '{LAUNCH_DATE}' AND '{ANALYSIS_DATE}'
GROUP BY 1, 2
ORDER BY 1, 2
"""
df_pred = run_query(q_pred, '3g Prediction value trends (rs.spending_summary)')
df_pred['date'] = pd.to_datetime(df_pred['date'])

display(df_pred.style.format({
    'i2a_pred':       '{:.6f}',
    'i2a_normalizer': '{:.6f}',
    'a2r_pred':       '{:.6f}',
    'a2r_normalizer': '{:.6f}',
}, na_rep='—'))

# ── Figure 1: I2A — wrapper_pred + wrapper_normalizer (all campaigns) ─────────
colors = {'ANDROID CPA': '#636EFA', 'IOS CPA': '#EF553B', 'ANDROID ROAS': '#00CC96'}
groups = ['ANDROID CPA', 'IOS CPA', 'ANDROID ROAS']

fig1 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['I2A Prediction (wrapper_pred)', 'I2A Normalizer (wrapper_normalizer)'],
    vertical_spacing=0.12
)
for grp in groups:
    d = df_pred[df_pred['campaign_group'] == grp].sort_values('date')
    if d.empty:
        continue
    clr = colors[grp]
    fig1.add_scatter(x=d['date'], y=d['i2a_pred'],
                     name=grp, line_color=clr, mode='lines+markers', marker=dict(size=5),
                     legendgroup=grp, showlegend=True, row=1, col=1)
    fig1.add_scatter(x=d['date'], y=d['i2a_normalizer'],
                     name=grp, line_color=clr, mode='lines+markers', marker=dict(size=5),
                     legendgroup=grp, showlegend=False, row=2, col=1)

fig1.update_layout(
    height=520,
    legend=dict(orientation='h', y=1.05, x=0),
    title='3g-① I2A Prediction & Normalizer — All Campaigns · KOR since launch'
)
fig1.show()
fig1.write_image(os.path.join(CHART_DIR, '3g_i2a_pred_trend.png'), scale=2)

# ── Figure 2: A2R — a2r_pred + a2r_normalizer (ROAS campaign only) ───────────
d_roas = df_pred[df_pred['campaign_group'] == 'ANDROID ROAS'].sort_values('date')

if d_roas['a2r_pred'].isna().all():
    print('⚠️  A2R pred: all NULL for ROAS campaign — frozen normalizer or no revenue signal yet')
else:
    fig2 = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=['A2R Prediction (a2r_pred)', 'A2R Normalizer (a2r_normalizer)'],
        vertical_spacing=0.12
    )
    clr = '#00CC96'
    fig2.add_scatter(x=d_roas['date'], y=d_roas['a2r_pred'],
                     name='ANDROID ROAS', line_color=clr, mode='lines+markers', marker=dict(size=5),
                     showlegend=True, row=1, col=1)
    fig2.add_scatter(x=d_roas['date'], y=d_roas['a2r_normalizer'],
                     name='ANDROID ROAS', line_color=clr, mode='lines+markers', marker=dict(size=5),
                     showlegend=False, row=2, col=1)

    fig2.update_layout(
        height=520,
        legend=dict(orientation='h', y=1.05, x=0),
        title='3g-② A2R Prediction & Normalizer — ROAS Campaign (iyH1) · KOR since launch'
    )
    fig2.show()
    fig2.write_image(os.path.join(CHART_DIR, '3g_a2r_pred_trend.png'), scale=2)


✅ 3g Prediction value trends (rs.spending_summary): 53 rows


,date,campaign_group,i2a_pred,i2a_normalizer,a2r_pred,a2r_normalizer
0,2026-03-24 00:00:00,ANDROID CPA,0.183380,0.591832,—,—
1,2026-03-24 00:00:00,IOS CPA,0.270100,0.352174,—,—
2,2026-03-25 00:00:00,ANDROID CPA,0.186674,0.591832,—,—
3,2026-03-25 00:00:00,IOS CPA,0.271017,0.352174,—,—
4,2026-03-26 00:00:00,ANDROID CPA,0.202122,0.591832,—,—
5,2026-03-26 00:00:00,IOS CPA,0.275807,0.352174,—,—
6,2026-03-27 00:00:00,ANDROID CPA,0.193861,0.591832,—,—
7,2026-03-27 00:00:00,IOS CPA,0.270494,0.352174,—,—
8,2026-03-28 00:00:00,ANDROID CPA,0.189847,0.591832,—,—
9,2026-03-28 00:00:00,IOS CPA,0.271193,0.352174,—,—


---
## Section 4 — Audience Saturation Check (KOR)

In [110]:
# fact_supply: bid_rate, win_rate, clear_rate, CPM trend
q_supply = f"""
SELECT
  date_utc,
  SUM(bid_requests)                                  AS bid_requests,
  SUM(bids)                                          AS bids,
  SUM(bids_won)                                      AS bids_won,
  SUM(impressions)                                   AS impressions,
  SUM(media_cost_usd)                                AS spend,
  SAFE_DIVIDE(SUM(bids), SUM(bid_requests))          AS bid_rate,
  SAFE_DIVIDE(SUM(bids_won), SUM(bids))              AS win_rate,
  SAFE_DIVIDE(SUM(impressions), SUM(bids_won))       AS clear_rate,
  SAFE_DIVIDE(SUM(impressions), SUM(bids))           AS bid_to_imp_rate,
  SAFE_DIVIDE(SUM(media_cost_usd), SUM(impressions)) * 1000 AS cpm
FROM `moloco-ae-view.athena.fact_supply`
WHERE campaign_id IN {CAMPAIGN_SQL}
  AND req.country = '{KOR}'
  AND date_utc >= '{LAUNCH_DATE}'
  AND date_utc < '{ANALYSIS_DATE}'
GROUP BY 1 ORDER BY 1
"""
df_supply = run_query(q_supply, '4 Audience saturation (KOR, fact_supply)')
df_supply['date_utc'] = pd.to_datetime(df_supply['date_utc'])

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['CPM (USD) — Rising = increasing competition',
                    'Bid-to-imp Rate — Falling = harder to convert bids into impressions'])
fig.add_scatter(x=df_supply['date_utc'], y=df_supply['cpm'],
                name='CPM', line_color='#EF553B', row=1, col=1)
fig.add_scatter(x=df_supply['date_utc'], y=df_supply['bid_to_imp_rate'],
                name='Bid-to-imp rate', line_color='#636EFA', row=2, col=1)
fig.add_scatter(x=df_supply['date_utc'], y=df_supply['win_rate'],
                name='Win rate', line_color='#AB63FA', line_dash='dash', row=2, col=1)
fig.add_scatter(x=df_supply['date_utc'], y=df_supply['clear_rate'],
                name='Clear rate', line_color='#00CC96', line_dash='dot', row=2, col=1)
fig.update_layout(height=520,
    title='4 — Audience Saturation: CPM trend + Bid-to-imp decomposition (KOR, since launch)')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '4_saturation_cpm_bid_to_imp_kor.png'), scale=2)

✅ 4 Audience saturation (KOR, fact_supply): 22 rows


---
## Section 6 — ML Calibration: ROAS Campaign Cold-Start Analysis

**Campaign:** `iyH1zVvUZpViudzo` (NEWnanaori_launch_moloco_KR_And_tROAS_260406)
**Launched:** 2026-04-06 · **Benchmark:** `nazpxG3J5MareHRz` (stonekey KOR Android ROAS, 6 weeks old)
**Context (CRU report ODSB-17637):** D7 ROAS = 1.05% after 8 days. Root cause: A2R normalizer frozen, pred calibrator not yet active (Day 15 = Apr 21), zero A2R calibration data. Bid CPMs $0.05–$0.08 vs exchange floors $0.50–$2.00 → 99.8% of bids rejected by FilterByBidfloor.

**Pred calibrator activation:** `HISTORY_AOP_CALIBRATOR_ACTIVATION_WINDOW = 15` ([`marvel2/…/ml_pred_calibrator_generation/query.py`](https://github.com/moloco/marvel2/blob/main/python/moloco-scripts/moloco/scripts/ml_pred_calibrator_generation/query.py), lines ~7–11) → activates **2026-04-21**.

In [129]:
# ── ROAS campaign params ─────────────────────────────────────────────────────
ROAS_CAMPAIGN       = 'iyH1zVvUZpViudzo'
BENCHMARK_CAMPAIGN  = 'nazpxG3J5MareHRz'   # stonekey KOR Android ROAS (6 weeks, calibrated)
ROAS_LAUNCH_DATE    = '2026-04-06'
PRED_CALIBRATOR_DATE = '2026-04-21'        # Day 15 — pred calibrator activation expected
ROAS_ANALYSIS_END   = ANALYSIS_DATE        # use global ANALYSIS_DATE from Section 0

### 6a — Model Predictions & Normalizer Trend

In [112]:
# Source: moloco-data-prod.younghan.spending_summary
# Confirmed column names (no avg_ prefix):
#   Preds: core_pred, wrapper_pred, a2r_pred
#   Normalizers: wrapper_normalizer, a2r_normalizer (bare 'normalizer' does NOT exist)
#   TCM: tcm | Spend: spent | Budget: budget
q_normalizer = f"""
SELECT
  date,
  campaign,
  ROUND(AVG(core_pred),           8) AS core_pred,
  ROUND(AVG(wrapper_pred),        8) AS wrapper_pred,
  ROUND(AVG(wrapper_normalizer),  8) AS wrapper_normalizer,
  ROUND(AVG(a2r_pred),            8) AS a2r_pred,
  ROUND(AVG(a2r_normalizer),      8) AS a2r_normalizer,
  ROUND(AVG(tcm),                 4) AS tcm
FROM `moloco-data-prod.younghan.spending_summary`
WHERE campaign IN ('{ROAS_CAMPAIGN}', '{BENCHMARK_CAMPAIGN}')
  AND date BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
GROUP BY 1, 2
ORDER BY 1, 2
"""
df_norm = run_query(q_normalizer, '6a Model predictions & normalizer trend')
display(df_norm.style.format({
    'core_pred': '{:.6f}', 'wrapper_pred': '{:.6f}', 'wrapper_normalizer': '{:.6f}',
    'a2r_pred': '{:.6f}', 'a2r_normalizer': '{:.6f}', 'tcm': '{:.3f}'
}, na_rep='—'))

# ⚠️ Expected: a2r_normalizer frozen for iyH1 (zero variance — no revenue signal to calibrate from)
# Benchmark stonekey: a2r_normalizer regularly updated on retraining

✅ 6a Model predictions & normalizer trend: 19 rows


,date,campaign,core_pred,wrapper_pred,wrapper_normalizer,a2r_pred,a2r_normalizer,tcm
0,2026-04-06,iyH1zVvUZpViudzo,0.000092,0.013120,0.013229,12.610955,15.242939,1.344
1,2026-04-06,nazpxG3J5MareHRz,0.000103,0.040461,0.200163,17.168888,30.019340,67.533
2,2026-04-07,iyH1zVvUZpViudzo,0.000073,0.011340,0.012482,12.013140,15.242939,1.652
3,2026-04-07,nazpxG3J5MareHRz,0.000096,0.038141,0.146734,15.527635,29.934021,67.003
4,2026-04-08,iyH1zVvUZpViudzo,0.000060,0.012123,0.013479,12.008765,15.242939,1.994
5,2026-04-08,nazpxG3J5MareHRz,0.000104,0.040079,0.139392,14.949068,29.934021,46.662
6,2026-04-09,iyH1zVvUZpViudzo,0.000061,0.015025,0.018509,12.848912,15.242939,3.378
7,2026-04-09,nazpxG3J5MareHRz,0.000115,0.047702,0.112909,15.734760,29.934021,26.803
8,2026-04-10,iyH1zVvUZpViudzo,0.000061,0.021021,0.028688,13.159081,15.242939,3.342
9,2026-04-10,nazpxG3J5MareHRz,0.000103,0.052615,0.121022,16.363656,29.934021,28.874


In [113]:
# 6a viz — A2R normalizer comparison: nanaori (frozen) vs stonekey (calibrated)
campaigns = [ROAS_CAMPAIGN, BENCHMARK_CAMPAIGN]
labels = {ROAS_CAMPAIGN: 'iyH1 (nanaori ROAS, 8d)', BENCHMARK_CAMPAIGN: 'stonekey ROAS (6wk)'}
colors = {ROAS_CAMPAIGN: '#EF553B', BENCHMARK_CAMPAIGN: '#636EFA'}

fig = make_subplots(rows=2, cols=2, shared_xaxes=False,
    subplot_titles=['A2R Normalizer (frozen = red flag)',
                    'A2R Prediction (daily avg)',
                    'TCM — budget multiplier escalation',
                    'Wrapper Normalizer'])

for cid in campaigns:
    d = df_norm[df_norm['campaign'] == cid].copy()
    d['date'] = pd.to_datetime(d['date'])
    lbl = labels[cid]
    clr = colors[cid]
    for row, col, metric in [(1,1,'a2r_normalizer'),(1,2,'a2r_pred'),(2,1,'tcm'),(2,2,'wrapper_normalizer')]:
        if metric in d.columns:
            fig.add_scatter(x=d['date'], y=d[metric], name=lbl, line_color=clr,
                            legendgroup=cid, showlegend=(row==1 and col==1), row=row, col=col)

# Use add_shape instead of add_vline — more reliable with datetime subplots
# xref='x3' targets the 3rd x-axis (row=2, col=1 = TCM subplot)
for xref in ['x', 'x2', 'x3', 'x4']:
    fig.add_shape(type='line',
        x0=PRED_CALIBRATOR_DATE, x1=PRED_CALIBRATOR_DATE,
        y0=0, y1=1, yref='paper',
        xref=xref,
        line=dict(dash='dash', color='green', width=1.5))
fig.add_annotation(
    x=PRED_CALIBRATOR_DATE, y=1.02, xref='x3', yref='paper',
    text='Calibrator ON<br>(Apr 21)', showarrow=False,
    font=dict(color='green', size=10))

fig.update_layout(height=600,
    title='6a — Model Predictions & Normalizer: iyH1 (cold) vs stonekey (calibrated)')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '6a_normalizer_trend.png'), scale=2)

### 6b — I2I Install Model Calibration

In [114]:
# Source: moloco-data-prod.ml_calibration_check.validation_i2i
# calibration_ratio = actual_installs / predicted_installs
# ratio < 1 = model over-predicts installs; ratio > 1 = under-predicts
q_i2i = f"""
SELECT
  date_imp,
  campaign,
  SUM(cnt_imp)            AS total_imps,
  SUM(cnt_install)        AS actual_installs,
  SUM(predicted_install)  AS predicted_installs,
  ROUND(SAFE_DIVIDE(SUM(cnt_install), SUM(predicted_install)), 4) AS calibration_ratio
FROM `moloco-data-prod.ml_calibration_check.validation_i2i`
WHERE campaign IN ('{ROAS_CAMPAIGN}', '{BENCHMARK_CAMPAIGN}')
  AND date_imp BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
GROUP BY 1, 2 ORDER BY 2, 1
"""
df_i2i = run_query(q_i2i, '6b I2I install model calibration')
display(df_i2i.style.format({
    'total_imps': '{:,.0f}', 'actual_installs': '{:,.0f}',
    'predicted_installs': '{:.1f}', 'calibration_ratio': '{:.3f}'
}, na_rep='—'))
print('\nInterpretation:')
print('  ratio 0.27-0.71 (iyH1): model over-predicts installs by 2-4× — inflates bid values')
print('  ratio 0.69-1.54 (stonekey): well-calibrated, centered near 1.0')

✅ 6b I2I install model calibration: 16 rows


,date_imp,campaign,total_imps,actual_installs,predicted_installs,calibration_ratio
0,2026-04-06,iyH1zVvUZpViudzo,"295,010",9,33.3,0.271
1,2026-04-07,iyH1zVvUZpViudzo,"756,070",18,60.9,0.295
2,2026-04-08,iyH1zVvUZpViudzo,"538,734",13,32.1,0.405
3,2026-04-09,iyH1zVvUZpViudzo,"1,058,886",14,40.9,0.342
4,2026-04-10,iyH1zVvUZpViudzo,"820,205",12,27.0,0.445
5,2026-04-11,iyH1zVvUZpViudzo,"834,883",18,25.4,0.708
6,2026-04-12,iyH1zVvUZpViudzo,"1,211,035",10,24.8,0.404
7,2026-04-13,iyH1zVvUZpViudzo,"814,571",12,15.5,0.773
8,2026-04-06,nazpxG3J5MareHRz,"4,400,810",59,73.2,0.806
9,2026-04-07,nazpxG3J5MareHRz,"2,543,338",44,42.4,1.038



Interpretation:
  ratio 0.27-0.71 (iyH1): model over-predicts installs by 2-4× — inflates bid values
  ratio 0.69-1.54 (stonekey): well-calibrated, centered near 1.0


In [115]:
# 6b viz — I2I calibration ratio over time (nanaori vs stonekey)
fig = go.Figure()
for cid, lbl, clr in [(ROAS_CAMPAIGN, 'iyH1 (nanaori)', '#EF553B'),
                       (BENCHMARK_CAMPAIGN, 'stonekey', '#636EFA')]:
    d = df_i2i[df_i2i['campaign'] == cid].sort_values('date_imp')
    if not d.empty:
        fig.add_scatter(x=pd.to_datetime(d['date_imp']), y=d['calibration_ratio'],
                        name=lbl, line_color=clr, mode='lines+markers')

fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='Perfect calibration (ratio = 1.0)')
fig.add_hrect(y0=0.8, y1=1.2, fillcolor='lightgreen', opacity=0.15,
              annotation_text='Healthy zone (0.8-1.2)', annotation_position='top left')
fig.update_layout(height=380, yaxis_title='Calibration ratio (actual / predicted)',
    title='6b — I2I Install Model Calibration: iyH1 vs stonekey benchmark')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '6b_i2i_calibration.png'), scale=2)

### 6c — A2R Revenue Model Calibration

In [116]:
# Source: moloco-data-prod.ml_calibration_check.validation_a2r_online
# Expected: ZERO rows for iyH1 (no revenue signal → model cannot calibrate)
# Benchmark stonekey: calibration_ratio 0.56-1.29 (active learning)
q_a2r = f"""
SELECT
  date_install,
  campaign,
  SUM(cnt_action)             AS total_actions,
  SUM(cnt_revenue)            AS total_revenue_events,
  ROUND(SUM(actual_revenue_usd),    4) AS actual_revenue,
  ROUND(SUM(predicted_revenue_usd), 4) AS predicted_revenue,
  ROUND(SAFE_DIVIDE(SUM(actual_revenue_usd),
                    SUM(predicted_revenue_usd)), 4) AS calibration_ratio
FROM `moloco-data-prod.ml_calibration_check.validation_a2r_online`
WHERE campaign IN ('{ROAS_CAMPAIGN}', '{BENCHMARK_CAMPAIGN}')
  AND date_install BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
GROUP BY 1, 2 ORDER BY 2, 1
"""
df_a2r = run_query(q_a2r, '6c A2R revenue model calibration')

if df_a2r[df_a2r['campaign'] == ROAS_CAMPAIGN].empty:
    print(f'\u274c {ROAS_CAMPAIGN}: ZERO rows — A2R model has no revenue signal to calibrate from')
else:
    print(f'\u2705 {ROAS_CAMPAIGN}: {len(df_a2r[df_a2r["campaign"] == ROAS_CAMPAIGN])} rows found')

display(df_a2r.style.format({
    'actual_revenue': '${:.4f}', 'predicted_revenue': '${:.4f}', 'calibration_ratio': '{:.3f}'
}, na_rep='—'))

✅ 6c A2R revenue model calibration: 3 rows
✅ iyH1zVvUZpViudzo: 1 rows found


,date_install,campaign,total_actions,total_revenue_events,actual_revenue,predicted_revenue,calibration_ratio
0,2026-04-07,iyH1zVvUZpViudzo,2,4,$8.0330,$48.2378,0.167
1,2026-04-06,nazpxG3J5MareHRz,10,87,$332.2078,$290.9648,1.142
2,2026-04-07,nazpxG3J5MareHRz,17,133,$328.5370,$436.5668,0.752


### 6d — Pred Calibrator Status

In [117]:
# Source: focal-elf-631.prod_stream_sampled.pricing_1to100
# HISTORY_AOP_CALIBRATOR_ACTIVATION_WINDOW = 15
# marvel2/python/moloco-scripts/moloco/scripts/ml_pred_calibrator_generation/query.py (lines ~7-11)
# Expected: calibration multiplier NULL or 1.0 until Apr 21 (Day 15)
q_calibrator = f"""
SELECT
  DATE(timestamp)                                                       AS date,
  ROUND(AVG(cand.multipliers.calibration), 6)                          AS avg_calibration_mult,
  COUNTIF(cand.multipliers.calibration IS NOT NULL
    AND cand.multipliers.calibration != 0
    AND cand.multipliers.calibration != 1.0)                           AS active_calibration_count,
  COUNT(*)                                                              AS total_candidates
FROM `focal-elf-631.prod_stream_sampled.pricing_1to100`,
  UNNEST(pricing.candidates) AS cand
WHERE DATE(timestamp) BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
  AND cand.campaign_id = '{ROAS_CAMPAIGN}'
GROUP BY 1 ORDER BY 1
"""
df_calibrator = run_query(q_calibrator, '6d Pred calibrator status')
display(df_calibrator.style.format({
    'avg_calibration_mult': '{:.6f}', 'active_calibration_count': '{:,.0f}',
    'total_candidates': '{:,.0f}'
}, na_rep='—'))
print(f'\nExpected: active_calibration_count = 0 until {PRED_CALIBRATOR_DATE}')
print('Each row where active_calibration_count > 0 = calibrator is live')

✅ 6d Pred calibrator status: 9 rows


,date,avg_calibration_mult,active_calibration_count,total_candidates
0,2026-04-06,—,0,"5,739"
1,2026-04-07,—,0,"9,001"
2,2026-04-08,—,0,"9,748"
3,2026-04-09,—,0,"9,490"
4,2026-04-10,—,0,"9,186"
5,2026-04-11,—,0,"8,773"
6,2026-04-12,—,0,"9,184"
7,2026-04-13,—,0,"9,995"
8,2026-04-14,—,0,"2,323"



Expected: active_calibration_count = 0 until 2026-04-21
Each row where active_calibration_count > 0 = calibrator is live


---
## Section 7 — Supporting Evidence

Sub-sections: **7a** TCM Escalation + Budget Instability · **7b** Bid Funnel & FilterByBidfloor · **7c** Exchange Efficiency & NHN Deep-Dive · **7d** Payer Concentration & Revenue Timing

### 7a — TCM Escalation & Budget Instability

In [118]:
# TCM trend (budget pacing multiplier) — rising TCM = system under-spending, not over-spending
# Source: moloco-data-prod.younghan.spending_summary
# Actual columns: tcm, budget, spent (no avg_ prefix)
q_tcm = f"""
SELECT
  date,
  ROUND(AVG(tcm),    4) AS avg_tcm,
  ROUND(AVG(budget), 2) AS avg_daily_budget,
  ROUND(SUM(spent),  2) AS total_spent,
  ROUND(SAFE_DIVIDE(SUM(spent), AVG(budget)) * 100, 1) AS budget_util_pct
FROM `moloco-data-prod.younghan.spending_summary`
WHERE campaign = '{ROAS_CAMPAIGN}'
  AND date BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
GROUP BY 1 ORDER BY 1
"""
df_tcm = run_query(q_tcm, '7a TCM + budget utilization')

# Budget change timeline from entity_history
q_budget = f"""
SELECT
  DATE(timestamp)                                                                   AS date,
  TIMESTAMP_TRUNC(timestamp, HOUR)                                                  AS ts,
  CAST(JSON_VALUE(json_entity, '$.capper.user_budget_enforcer.daily_budget') AS FLOAT64) AS daily_budget
FROM `focal-elf-631.entity_history.prod_entity_history`
WHERE entity_type = 'RTB_CAMPAIGN'
  AND JSON_VALUE(json_entity, '$.name') = '{ROAS_CAMPAIGN}'
  AND DATE(timestamp) >= '{ROAS_LAUNCH_DATE}'
  AND JSON_VALUE(json_entity, '$.capper.user_budget_enforcer.daily_budget') IS NOT NULL
ORDER BY timestamp
"""
df_budget = run_query(q_budget, '7a Budget change history')
display(df_budget)
print(f'Budget changes: {len(df_budget)} snapshots — frequent changes hinder model learning')

✅ 7a TCM + budget utilization: 9 rows
✅ 7a Budget change history: 132 rows


,date,ts,daily_budget
0,2026-04-06,1775444400000000,10.00
1,2026-04-06,1775455200000000,980.07
2,2026-04-06,1775473200000000,980.07
3,2026-04-06,1775473200000000,980.07
4,2026-04-06,1775473200000000,980.07
...,...,...,...
127,2026-04-14,1776139200000000,653.38
128,2026-04-14,1776160800000000,653.38
129,2026-04-15,1776211200000000,653.38
130,2026-04-15,1776236400000000,653.38


Budget changes: 132 snapshots — frequent changes hinder model learning


In [119]:
# 7a viz — TCM escalation + budget changes overlaid
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['TCM (Budget Multiplier) — escalation signals under-spend',
                    'Daily Budget (USD) — instability hinders model learning'])
df_tcm['date'] = pd.to_datetime(df_tcm['date'])

fig.add_scatter(x=df_tcm['date'], y=df_tcm['avg_tcm'],
                name='TCM', line_color='#EF553B', mode='lines+markers', row=1, col=1)
fig.add_hline(y=1.0, line_dash='dot', line_color='gray',
              annotation_text='TCM=1 (normal)', row=1, col=1)

if not df_budget.empty:
    df_budget['ts'] = pd.to_datetime(df_budget['ts'])
    fig.add_scatter(x=df_budget['ts'], y=df_budget['daily_budget'],
                    name='Daily budget', line_color='#636EFA', mode='lines+markers',
                    line_shape='hv', row=2, col=1)

# Use add_shape for reliable datetime vline in subplots
for xref in ['x', 'x2']:
    fig.add_shape(type='line',
        x0=PRED_CALIBRATOR_DATE, x1=PRED_CALIBRATOR_DATE,
        y0=0, y1=1, yref='paper', xref=xref,
        line=dict(dash='dash', color='green', width=1.5))
fig.add_annotation(
    x=PRED_CALIBRATOR_DATE, y=1.02, xref='x2', yref='paper',
    text='Calibrator ON (Apr 21)', showarrow=False,
    font=dict(color='green', size=10))
fig.update_layout(height=500,
    title='7a — TCM Escalation & Budget Instability (iyH1 ROAS campaign)')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '7a_tcm_budget_instability.png'), scale=2)

### 7b — Bid Funnel & FilterByBidfloor

In [120]:
# Filter funnel — how many candidates pass each filter stage
# Source: moloco-data-prod.younghan.campaign_trace_raw_prod
q_funnel = f"""
SELECT
  reason_order,
  reason,
  exchange,
  ROUND(SUM(1/rate) / 1e6, 3) AS req_millions
FROM `moloco-data-prod.younghan.campaign_trace_raw_prod`
WHERE date BETWEEN DATE('{ROAS_LAUNCH_DATE}') AND DATE('{ROAS_ANALYSIS_END}')
  AND campaign = '{ROAS_CAMPAIGN}'
GROUP BY 1, 2, 3
ORDER BY exchange, reason_order DESC
"""
df_funnel = run_query(q_funnel, '7b Bid filter funnel by exchange')

# Summary: pass rate through FilterByBidfloor
q_bidfloor = f"""
SELECT
  DATE(timestamp)                                                     AS date,
  COUNTIF(cand.candidate_result = 'CommitBid')                       AS commit_bids,
  COUNTIF(cand.candidate_result != 'CommitBid')                      AS filtered_out,
  COUNT(*)                                                            AS total_candidates,
  ROUND(SAFE_DIVIDE(
    COUNTIF(cand.candidate_result = 'CommitBid'), COUNT(*)) * 100, 2) AS commit_pct,
  ROUND(AVG(cand.bid_price) / 1e6, 4)                                AS avg_bid_cpm_usd
FROM `focal-elf-631.prod_stream_sampled.pricing_1to100`,
  UNNEST(pricing.candidates) AS cand
WHERE DATE(timestamp) BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
  AND cand.campaign_id = '{ROAS_CAMPAIGN}'
GROUP BY 1 ORDER BY 1
"""
df_bidfloor = run_query(q_bidfloor, '7b Bid price vs floor (pricing stream)')
display(df_bidfloor.style.format({
    'commit_bids': '{:,.0f}', 'filtered_out': '{:,.0f}', 'total_candidates': '{:,.0f}',
    'commit_pct': '{:.2f}%', 'avg_bid_cpm_usd': '${:.4f}'
}, na_rep='—'))
print('\nExpected: commit_pct ~0.2% (99.8% rejection); avg_bid_cpm ~$0.05-$0.08 vs floors $0.50-$2.00')

✅ 7b Bid filter funnel by exchange: 328 rows
✅ 7b Bid price vs floor (pricing stream): 9 rows


,date,commit_bids,filtered_out,total_candidates,commit_pct,avg_bid_cpm_usd
0,2026-04-06,2,"5,614","5,616",0.04%,$0.0001
1,2026-04-07,24,"9,283","9,307",0.26%,$0.0001
2,2026-04-08,10,"9,803","9,813",0.10%,$0.0001
3,2026-04-09,22,"9,545","9,567",0.23%,$0.0002
4,2026-04-10,16,"10,410","10,426",0.15%,$0.0001
5,2026-04-11,16,"8,618","8,634",0.19%,$0.0001
6,2026-04-12,30,"10,241","10,271",0.29%,$0.0002
7,2026-04-13,18,"9,076","9,094",0.20%,$0.0002
8,2026-04-14,4,"2,296","2,300",0.17%,$0.0001



Expected: commit_pct ~0.2% (99.8% rejection); avg_bid_cpm ~$0.05-$0.08 vs floors $0.50-$2.00


### 7c — Exchange Efficiency & NHN Deep-Dive

In [121]:
# Exchange-level spend, installs, revenue for the ROAS campaign
q_exchange = f"""
SELECT
  exchange,
  SUM(gross_spend_usd)                                              AS spend,
  SUM(impressions)                                                  AS impressions,
  SUM(installs)                                                     AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))                  AS cpi,
  SUM(revenue_d7)                                                   AS revenue_d7,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd))                AS roas_d7,
  SAFE_DIVIDE(SUM(gross_spend_usd),
    SUM(SUM(gross_spend_usd)) OVER ()) * 100                        AS spend_share_pct
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id = '{ROAS_CAMPAIGN}'
  AND date_utc BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
GROUP BY 1
ORDER BY spend DESC
"""
df_exchange = run_query(q_exchange, '7c Exchange efficiency (iyH1 ROAS campaign)')
display(df_exchange.style.format({
    'spend': '${:,.0f}', 'impressions': '{:,.0f}', 'installs': '{:,.0f}',
    'cpi': '${:.2f}', 'revenue_d7': '${:.2f}', 'roas_d7': '{:.2%}',
    'spend_share_pct': '{:.1f}%'
}, na_rep='—'))

# Flag NHN
nhn = df_exchange[df_exchange['exchange'] == 'NHN']
if not nhn.empty:
    r = nhn.iloc[0]
    print(f'\n\u26a0\ufe0f  NHN: {r["spend_share_pct"]:.1f}% of spend (${r["spend"]:,.0f}), '
          f'{r["installs"]:.0f} installs (${r["cpi"]:.0f} CPI), '
          f'D7 revenue = ${r["revenue_d7"]:.2f} — recommend budget cap or exclusion')

✅ 7c Exchange efficiency (iyH1 ROAS campaign): 33 rows


,exchange,spend,impressions,installs,cpi,revenue_d7,roas_d7,spend_share_pct
0,NHN,"$1,986","373,410",12,$165.47,$0.00,0.00%,31.4%
1,KAKAO,"$1,313","2,799,567",67,$19.60,$13.15,1.00%,20.7%
2,MANPLUS,"$1,303","187,901",10,$130.34,$37.11,2.85%,20.6%
3,TPMN,$858,"1,133,440",10,$85.81,$2.74,0.32%,13.6%
4,ADX_RTB,$165,"557,744",4,$41.17,$7.13,4.33%,2.6%
5,ADPIE,$148,"713,660",1,$147.73,$0.00,0.00%,2.3%
6,MOLOCO_SDK_MAX,$116,"137,294",6,$19.35,$0.00,0.00%,1.8%
7,BIDENCE,$107,"28,315",0,—,$0.00,0.00%,1.7%
8,INMOBI,$93,"115,388",6,$15.43,$0.00,0.00%,1.5%
9,APPLOVIN,$46,"110,787",1,$46.20,$0.00,0.00%,0.7%



⚠️  NHN: 31.4% of spend ($1,986), 12 installs ($165 CPI), D7 revenue = $0.00 — recommend budget cap or exclusion


In [122]:
# 7c viz — Exchange spend share vs revenue (bar + scatter)
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Spend Share by Exchange (%)', 'CPI vs D7 ROAS by Exchange'])

df_ex_top = df_exchange.head(8)
colors_ex = ['#EF553B' if x == 'NHN' else '#636EFA' for x in df_ex_top['exchange']]

fig.add_bar(x=df_ex_top['exchange'], y=df_ex_top['spend_share_pct'],
            marker_color=colors_ex, name='Spend share %',
            text=df_ex_top['spend_share_pct'].map('{:.1f}%'.format),
            textposition='outside', row=1, col=1)
fig.add_scatter(x=df_ex_top['cpi'], y=df_ex_top['roas_d7'] * 100,
                text=df_ex_top['exchange'], mode='markers+text',
                marker=dict(size=df_ex_top['spend_share_pct'].clip(lower=3) * 2,
                            color=colors_ex),
                textposition='top center', name='CPI vs ROAS', row=1, col=2)
fig.update_yaxes(title_text='ROAS D7 (%)', row=1, col=2)
fig.update_xaxes(title_text='CPI (USD)', row=1, col=2)
fig.update_layout(height=420, showlegend=False,
    title='7c — Exchange Efficiency: iyH1 ROAS campaign (NHN = red)')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '7c_exchange_efficiency.png'), scale=2)

### 7d — Payer Concentration & Revenue Timing

In [123]:
# Payer concentration — whale risk (1 payer = large % of total revenue)
# Fix: removed event.name = 'revenue' filter — AF revenue postbacks use MMP-specific
# event names (af_purchase, PAY, etc.), not the literal string 'revenue'.
# Revenue presence is reliably identified by event.revenue_usd.amount > 0.
# Source: moloco-dsp-data-view.postback.pb
q_payers = f"""
SELECT
  device.idfa                        AS device_ifa,
  COUNT(*)                          AS revenue_events,
  SUM(event.revenue_usd.amount)     AS total_revenue_usd,
  MIN(event.revenue_usd.amount)     AS min_rev,
  MAX(event.revenue_usd.amount)     AS max_rev,
  MIN(DATE(timestamp))              AS first_event_date
FROM `moloco-dsp-data-view.postback.pb`
WHERE DATE(timestamp) BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
  AND moloco.campaign_id = '{ROAS_CAMPAIGN}'
  AND event.revenue_usd.amount > 0
  AND attribution.attributed = TRUE
GROUP BY 1
ORDER BY total_revenue_usd DESC
"""
df_payers = run_query(q_payers, '7d Payer concentration (iyH1 ROAS)')
total_rev = df_payers['total_revenue_usd'].sum()
print(f'Total payers: {len(df_payers)} | Total revenue: ${total_rev:.2f}')
if len(df_payers) > 0:
    top1_pct = df_payers.iloc[0]['total_revenue_usd'] / total_rev * 100
    print(f'Top 1 payer: ${df_payers.iloc[0]["total_revenue_usd"]:.2f} ({top1_pct:.0f}% of revenue)')
    print(f'⚠️  ROAS is statistically unreliable with {len(df_payers)} payers — need 500+ installs for meaningful signal')
display(df_payers.style.format({'total_revenue_usd': '${:.2f}', 'min_rev': '${:.2f}', 'max_rev': '${:.2f}'}, na_rep='—'))

✅ 7d Payer concentration (iyH1 ROAS): 9 rows
Total payers: 9 | Total revenue: $79.53
Top 1 payer: $37.11 (47% of revenue)
⚠️  ROAS is statistically unreliable with 9 payers — need 500+ installs for meaningful signal


,device_ifa,revenue_events,total_revenue_usd,min_rev,max_rev,first_event_date
0,6c4b5b9a-4dbd-4107-8079-9ccfbd25f542,5,$37.11,$0.47,$14.50,2026-04-11
1,516b2fce-66a5-4023-b8e2-dc0ea11b3aaa,2,$12.19,$2.33,$9.86,2026-04-15
2,5d6cd5e0-9674-420e-85f2-3ca3f6ab5475,2,$7.21,$2.31,$4.90,2026-04-11
3,b62ea7e8-3c14-4139-bc3c-41c5f586accf,2,$7.13,$2.29,$4.84,2026-04-08
4,98d38c5d-8157-4795-9043-56ece59912b8,2,$5.29,$0.46,$4.83,2026-04-07
5,0885a3b2-7ce6-4003-b5b4-9763ab4bc722,2,$2.79,$0.47,$2.31,2026-04-13
6,7daa9b4a-6c7a-41a4-aa7e-f4627d852ac1,2,$2.75,$0.46,$2.29,2026-04-08
7,83937f19-26d0-4ed8-b62e-4172bc563fbd,2,$2.74,$0.46,$2.28,2026-04-07
8,36f61b15-a8a2-41a8-bd70-0ab51e68f87d,1,$2.32,$2.32,$2.32,2026-04-14


In [124]:
# Revenue timing — days from install to first revenue event
# Fix: removed event.name = 'revenue' — use event.revenue_usd.amount > 0 to identify revenue
q_rev_timing = f"""
SELECT
  DATE_DIFF(DATE(timestamp), DATE(event.install_at), DAY) AS days_to_revenue,
  COUNT(*)                                                  AS event_count,
  SUM(event.revenue_usd.amount)                            AS revenue_usd
FROM `moloco-dsp-data-view.postback.pb`
WHERE DATE(timestamp) BETWEEN '{ROAS_LAUNCH_DATE}' AND '{ROAS_ANALYSIS_END}'
  AND moloco.campaign_id = '{ROAS_CAMPAIGN}'
  AND attribution.attributed = TRUE
  AND event.revenue_usd.amount > 0
GROUP BY 1
ORDER BY 1
"""
df_timing = run_query(q_rev_timing, '7d Revenue timing (install-to-revenue delay)')

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Revenue Events by Day Since Install', 'Cumulative Revenue by Day'])
if not df_timing.empty:
    df_timing['cumulative_rev'] = df_timing['revenue_usd'].cumsum()
    fig.add_bar(x=df_timing['days_to_revenue'], y=df_timing['event_count'],
                name='Revenue events', marker_color='#636EFA', row=1, col=1)
    fig.add_scatter(x=df_timing['days_to_revenue'], y=df_timing['cumulative_rev'],
                    name='Cumulative revenue', line_color='#EF553B', mode='lines+markers', row=1, col=2)
fig.update_xaxes(title_text='Days from install', row=1, col=1)
fig.update_xaxes(title_text='Days from install', row=1, col=2)
fig.update_layout(height=380, title='7d — Revenue Timing: Install-to-Revenue Delay (iyH1 ROAS)')
fig.show()
fig.write_image(os.path.join(CHART_DIR, '7d_revenue_timing.png'), scale=2)

✅ 7d Revenue timing (install-to-revenue delay): 4 rows


---
## Section 8 — Summary & Prioritized Recommendations

*Populate after running all sections above.*

---

### Status Dashboard

```
[ ] CPI efficiency (KOR, Android / iOS)
[ ] D1 ROAS vs 5% KPI (KOR, Android / iOS)
[ ] IPM decay rate vs benchmark
[ ] Install-to-Login rate (download friction signal)
[ ] Kakao VT attribution quality
[ ] User quality vs organic baseline
[ ] Audience saturation risk
```

### Root Cause Hypothesis (pre-analysis, from Searchlight)

IPM collapse −75% (Android) −89% (iOS) W1→W3 — worst 2–3% of 540 KOR gaming campaigns. Root cause: audience exhaustion + D56 cold-start compounding creative selection randomness. Creative fatigue ruled out (F1). Kakao spend (30–44%) generating primarily VT installs (95.6%) with questionable incrementality.

### Prioritized Recommendations

*Fill in after running sections 2b, 2c, 2d, 2e.*

1. …
2. …
3. …
